In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse variable names for efficiency
metric = pd.read_csv('metric.csv')
trace = pd.read_csv('trace.csv')
log = pd.read_csv('log.csv')
error_logs = pd.read_csv('error_logs.csv')

def make_meta(df):
    meta = {}
    meta['total_rows'] = int(df.shape[0])
    if df.shape[0] == 0 or 'timestamp' not in df.columns:
        meta['min_ts'] = None
        meta['max_ts'] = None
    else:
        # Convert min/max using UTC+8 (Asia/Shanghai)
        min_ts = pd.to_datetime(int(df['timestamp'].min()), unit='s').tz_localize(tz)
        max_ts = pd.to_datetime(int(df['timestamp'].max()), unit='s').tz_localize(tz)
        meta['min_ts'] = min_ts
        meta['max_ts'] = max_ts
    return pd.DataFrame([meta])

def top_counts(df, col, topn=20):
    if col not in df.columns:
        return pd.DataFrame(columns=[col, 'count'])
    vc = df[col].astype(str).value_counts().head(topn)
    tdf = vc.reset_index()
    tdf.columns = [col, 'count']
    return tdf

# metric summary
metric_meta = make_meta(metric)
metric_top_cmdb = top_counts(metric, 'cmdb_id', 20)
metric_top_kpi = top_counts(metric, 'kpi_name', 20)

metric_summary = {
    'meta': metric_meta,
    'top_cmdb': metric_top_cmdb,
    'top_kpi_name': metric_top_kpi
}

# trace summary
trace_meta = make_meta(trace)
trace_top_cmdb = top_counts(trace, 'cmdb_id', 20)
trace_top_name = top_counts(trace, 'trace_name', 20)

trace_summary = {
    'meta': trace_meta,
    'top_cmdb': trace_top_cmdb,
    'top_trace_name': trace_top_name
}

# log summary
log_meta = make_meta(log)
log_top_cmdb = top_counts(log, 'cmdb_id', 20)
log_top_name = top_counts(log, 'log_name', 20)

log_summary = {
    'meta': log_meta,
    'top_cmdb': log_top_cmdb,
    'top_log_name': log_top_name
}

# error_logs summary - message samples first 5 chars for privacy
error_meta = make_meta(error_logs)
error_top_cmdb = top_counts(error_logs, 'cmdb_id', 20)
if 'message' in error_logs.columns and error_logs.shape[0] > 0:
    msg_samples = error_logs['message'].astype(str).str[:5]
    error_top_msg = msg_samples.value_counts().head(20).reset_index()
    error_top_msg.columns = ['msg_sample_first5', 'count']
else:
    error_top_msg = pd.DataFrame(columns=['msg_sample_first5', 'count'])

error_logs_summary = {
    'meta': error_meta,
    'top_cmdb': error_top_cmdb,
    'top_message_samples': error_top_msg
}

# Display compact summaries (limited to top 20 where applicable)
metric_summary, trace_summary, log_summary, error_logs_summary

```
Out[1]:
```
summary = (
    "Summary of telemetry files (times shown in UTC+8):\n\n"
    "metric.csv:\n"
    "- Total rows: 321,472\n"
    "- Time range: 2022-03-20 16:30 to 2022-03-20 17:30\n"
    "- Top cmdb_id (top 5): adservice (20,801), adservice2 (20,313), frontend2-0 (11,956), "
    "frontend-2 (11,468), frontend-1 (11,468)\n"
    "- Top kpi_name patterns: dominated by container.node-6 metrics (many entries with count 2,196), "
    "examples include container.node-6.container_network_transmit_*, container.node-6.container_network_receive_*, "
    "and numerous container.node-6 fs/memory metrics (several with count ~2,074).\n\n"
    "trace.csv:\n"
    "- Total rows: 29,584\n"
    "- Time range: 2022-03-20 17:00 to 2022-03-20 17:29\n"
    "- Top cmdb_id (top 5): frontend-0 (2,576), frontend-1 (2,572), frontend-2 (2,316), "
    "checkoutservice-1 (1,720), checkoutservice-0 (1,524)\n"
    "- Top trace_name (examples, top entries): trace.from_frontend-*.duration_mean / duration_p95 / error_rate / row_count "
    "(counts in the 500–600 range), and trace.self.* metrics (counts ~463).\n\n"
    "log.csv:\n"
    "- Total rows: 1,750\n"
    "- Time range: 2022-03-20 17:00 to 2022-03-20 17:30\n"
    "- Top cmdb_id: many service pods appear regularly (counts ~60–62 for many pods)\n"
    "- Top log_name: log.error_count (875), log.row_count (875)\n\n"
    "error_logs.csv:\n"
    "- Total rows: 0 (no entries)\n\n"
    "Notes: Top lists were taken from the available top-20 summaries; I kept examples concise and focused on the highest-frequency items."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

({'meta':    total_rows                    min_ts                    max_ts
0      321472 2022-03-20 16:30:00+08:00 2022-03-20 17:30:00+08:00, 'top_cmdb':                      cmdb_id  count
0                  adservice  20801
1                 adservice2  20313
2                frontend2-0  11956
3                 frontend-2  11468
4                 frontend-1  11468
5                 frontend-0  11468
6          checkoutservice-1   8784
7         checkoutservice2-0   8784
8          checkoutservice-2   8540
9          checkoutservice-0   8540
10   productcatalogservice-1   7320
11   productcatalogservice-0   7320
12   productcatalogservice-2   7076
13  productcatalogservice2-0   6344
14  recommendationservice2-0   6222
15             cartservice-2   6100
16             cartservice-0   6100
17            cartservice2-0   6100
18   recommendationservice-1   5978
19   recommendationservice-2   5978, 'top_kpi_name':                                              kpi_name  count
0   container.node-6.container_network_transmit_pa...   2196
1   container.node-6.container_network_transmit_er...   2196
2   container.node-6.container_network_transmit_pa...   2196
3   container.node-6.container_network_receive_MB....   2196
4   container.node-6.container_network_receive_pac...   2196
5   container.node-6.container_network_receive_pac...   2196
6   container.node-6.container_network_receive_err...   2196
7   container.node-6.container_network_transmit_MB...   2196
8        container.node-6.container_fs_reads./dev/vda   2074
9   container.node-6.container_fs_read_seconds./de...   2074
10   container.node-6.container_fs_limit_MB./dev/vda1   2074
11              container.node-6.container_memory_rss   2074
12     container.node-6.container_memory_max_usage_MB   2074
13      container.node-6.container_memory_mapped_file   2074
14    container.node-6.container_fs_reads_MB./dev/vda   2074
15      container.node-6.container_fs_reads./dev/vda1   2074
16  container.node-6.container_fs_sector_writes./d...   2074
17  container.node-6.container_fs_reads_merged./de...   2074
18  container.node-6.container_fs_sector_reads./de...   2074
19  container.node-6.container_tasks_state.uninter...   2074}, {'meta':    total_rows                    min_ts                    max_ts
0       29584 2022-03-20 17:00:00+08:00 2022-03-20 17:29:00+08:00, 'top_cmdb':                     cmdb_id  count
0                frontend-0   2576
1                frontend-1   2572
2                frontend-2   2316
3         checkoutservice-1   1720
4         checkoutservice-0   1524
5         checkoutservice-2   1504
6               frontend2-0   1072
7   productcatalogservice-1   1072
8   productcatalogservice-0   1064
9   productcatalogservice-2   1060
10       checkoutservice2-0   1008
11  recommendationservice-1    840
12  recommendationservice-0    840
13  recommendationservice-2    840
14            cartservice-0    720
15            cartservice-2    708
16            cartservice-1    704
17        currencyservice-2    652
18        currencyservice-1    644
19        currencyservice-0    640, 'top_trace_name':                                     trace_name  count
0          trace.from_frontend-0.duration_mean    584
1           trace.from_frontend-0.duration_p95    584
2             trace.from_frontend-0.error_rate    584
3              trace.from_frontend-0.row_count    584
4          trace.from_frontend-1.duration_mean    583
5           trace.from_frontend-1.duration_p95    583
6             trace.from_frontend-1.error_rate    583
7              trace.from_frontend-1.row_count    583
8          trace.from_frontend-2.duration_mean    519
9           trace.from_frontend-2.duration_p95    519
10            trace.from_frontend-2.error_rate    519
11             trace.from_frontend-2.row_count    519
12                        trace.self.row_count    463
13                       trace.self.error_rate    463
14                     trace.self.duration_p95    463
15                    trace.self.duration_mean    463
16  trace.from_checkoutservice-1.duration_mean    337
17   trace.from_checkoutservice-1.duration_p95    337
18     trace.from_checkoutservice-1.error_rate    337
19      trace.from_checkoutservice-1.row_count    337}, {'meta':    total_rows                    min_ts                    max_ts
0        1750 2022-03-20 17:00:00+08:00 2022-03-20 17:30:00+08:00, 'top_cmdb':                     cmdb_id  count
0             cartservice-2     62
1             cartservice-1     62
2             cartservice-0     62
3                frontend-0     62
4         currencyservice-2     62
5         currencyservice-1     62
6         currencyservice-0     62
7         shippingservice-1     62
8   recommendationservice-1     62
9   recommendationservice-0     62
10        shippingservice-0     62
11  recommendationservice-2     62
12               frontend-1     62
13        checkoutservice-1     60
14              adservice-0     60
15  productcatalogservice-0     60
16        shippingservice-2     60
17  productcatalogservice-2     60
18               frontend-2     60
19              adservice-2     60, 'top_log_name':           log_name  count
0  log.error_count    875
1    log.row_count    875}, {'meta':    total_rows min_ts max_ts
0           0   None   None, 'top_cmdb': Empty DataFrame
Columns: [cmdb_id, count]
Index: [], 'top_message_samples': Empty DataFrame
Columns: [msg_sample_first5, count]
Index: []})```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing DataFrames if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')
try:
    log
except NameError:
    log = pd.read_csv('log.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp -> timezone-aware datetime column 'ts' (UTC+8)
if 'ts' not in metric.columns:
    metric['ts'] = pd.to_datetime(metric['timestamp'], unit='s').dt.tz_localize(tz)
if 'ts' not in trace.columns:
    trace['ts'] = pd.to_datetime(trace['timestamp'], unit='s').dt.tz_localize(tz)
if 'ts' not in log.columns:
    log['ts'] = pd.to_datetime(log['timestamp'], unit='s').dt.tz_localize(tz)

# Coerce value to numeric where applicable (non-numeric -> NaN)
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')
log['value'] = pd.to_numeric(log['value'], errors='coerce')

# Incident window in UTC+8
incident_start = tz.localize(pd.to_datetime("2022-03-21 01:00:00"))
incident_end = tz.localize(pd.to_datetime("2022-03-21 01:30:00"))

# Helper to aggregate: compute global p95, p5, total_count, and incident_count
def aggregate_series(df, group_cols, value_col='value'):
    # Global aggregates (percentiles computed on full series per group)
    grp = df.groupby(group_cols)[value_col].agg(
        total_count = 'size',
        p95 = lambda x: x.quantile(0.95) if x.notna().any() else float('nan'),
        p5  = lambda x: x.quantile(0.05) if x.notna().any() else float('nan')
    ).reset_index()
    # Incident window counts
    in_win = df[(df['ts'] >= incident_start) & (df['ts'] <= incident_end)]
    inc_counts = in_win.groupby(group_cols).size().reset_index(name='incident_count')
    # Merge and fill missing incident_count with 0
    merged = grp.merge(inc_counts, on=group_cols, how='left')
    merged['incident_count'] = merged['incident_count'].fillna(0).astype(int)
    # Sort by total_count desc and keep top 20 for compactness
    merged = merged.sort_values('total_count', ascending=False).reset_index(drop=True)
    return merged

# 1) metric.csv aggregation by (cmdb_id, kpi_name)
metric_agg = aggregate_series(metric, ['cmdb_id', 'kpi_name'])
metric_top20 = metric_agg.head(20)[['cmdb_id','kpi_name','total_count','p95','p5','incident_count']]

# 2) trace.csv aggregation by (cmdb_id, trace_name)
trace_agg = aggregate_series(trace, ['cmdb_id', 'trace_name'])
trace_top20 = trace_agg.head(20)[['cmdb_id','trace_name','total_count','p95','p5','incident_count']]

# 3) log.csv aggregation by (cmdb_id, log_name)
log_agg = aggregate_series(log, ['cmdb_id', 'log_name'])
log_top20 = log_agg.head(20)[['cmdb_id','log_name','total_count','p95','p5','incident_count']]

# 4) Single-line summary whether any rows exist in any file within the incident window
metric_inc_count = int(((metric['ts'] >= incident_start) & (metric['ts'] <= incident_end)).sum())
trace_inc_count  = int(((trace['ts']  >= incident_start) & (trace['ts']  <= incident_end)).sum())
log_inc_count    = int(((log['ts']    >= incident_start) & (log['ts']    <= incident_end)).sum())

any_exist = 'yes' if (metric_inc_count + trace_inc_count + log_inc_count) > 0 else 'no'
incident_summary = pd.DataFrame({
    'file': ['metric.csv','trace.csv','log.csv'],
    'incident_rows': [metric_inc_count, trace_inc_count, log_inc_count]
})
incident_one_line = f"Any rows in incident window? {any_exist}. Counts per file: metric.csv={metric_inc_count}, trace.csv={trace_inc_count}, log.csv={log_inc_count}"

# Return compact results (variables stored for follow-ups)
metric_top20, trace_top20, log_top20, incident_summary, incident_one_line

```
Out[2]:
```
Summary (plain English):

- metric.csv (aggregated by cmdb_id, kpi_name):
  - Top entries (by total_count) are dominated by shippingservice2-0 metrics — each top pair has total_count = 61.
  - Example p95/p5 ranges for those top pairs: p95 values vary (e.g., 946,176; 128.7266; 10,002,430; 0.4625; 0.0355) and corresponding p5 values shown in the table; all incident_count = 0 for the top 20 pairs.
  - In short: top pairs have 61 points in their full series, reasonable p95/p5 values, and none have points inside the incident window.

- trace.csv (aggregated by cmdb_id, trace_name):
  - Top entries have total_count = 30 (examples include shippingservice2-0 trace.from_frontend2-0.row_count with p95=12, p5=3; many adservice-* trace.from_frontend-* metrics with tiny duration/error_rate quantiles).
  - All top 20 trace groups have incident_count = 0.

- log.csv (aggregated by cmdb_id, log_name):
  - Top entries have total_count ≈ 31 (examples: cartservice-2 log.row_count total_count=31, p95≈313.0, p5≈241.5; many other services' log.row_count around p95≈300, p5≈190–250).
  - All top 20 log groups have incident_count = 0.

- Incident-window summary (2022-03-21 01:00 to 2022-03-21 01:30, UTC+8):
  - Any rows in the incident window? No.
  - Counts in window: metric.csv = 0, trace.csv = 0, log.csv = 0.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  total_count           p95            p5  incident_count
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....           61  1.094500e+03  5.472500e+02               0
1            adservice                                     app.grpc.count           61  8.700000e+01  6.700000e+01               0
2   shippingservice2-0  container.node-6.container_network_receive_MB....           61  3.550863e-02  2.607536e-02               0
3   shippingservice2-0  container.node-6.container_network_receive_err...           61  0.000000e+00  0.000000e+00               0
4   shippingservice2-0  container.node-6.container_network_receive_pac...           61  3.720000e+02  2.945000e+02               0
5   shippingservice2-0  container.node-6.container_network_receive_pac...           61  0.000000e+00  0.000000e+00               0
6   shippingservice2-0  container.node-6.container_network_transmit_MB...           61  4.625049e-01  4.366550e-01               0
7   shippingservice2-0  container.node-6.container_network_transmit_er...           61  0.000000e+00  0.000000e+00               0
8   shippingservice2-0  container.node-6.container_network_transmit_pa...           61  2.360000e+02  1.850000e+02               0
9   shippingservice2-0  container.node-6.container_network_transmit_pa...           61  0.000000e+00  0.000000e+00               0
10  shippingservice2-0  container.node-6.container_memory_failures.hie...           61  0.000000e+00  0.000000e+00               0
11  shippingservice2-0  container.node-6.container_memory_failures.hie...           61  0.000000e+00  0.000000e+00               0
12  shippingservice2-0      container.node-6.container_memory_mapped_file           61  9.461760e+05  6.758400e+05               0
13  shippingservice2-0     container.node-6.container_memory_max_usage_MB           61  1.287266e+02  1.287266e+02               0
14  shippingservice2-0              container.node-6.container_memory_rss           61  1.000243e+07  1.000243e+07               0
15  shippingservice2-0             container.node-6.container_memory_swap           61  0.000000e+00  0.000000e+00               0
16  shippingservice2-0         container.node-6.container_memory_usage_MB           61  3.201562e+01  3.193359e+01               0
17  shippingservice2-0   container.node-6.container_memory_working_set_MB           61  1.881250e+01  1.873047e+01               0
18  shippingservice2-0     container.node-6.container_fs_writes./dev/vda1           61  0.000000e+00  0.000000e+00               0
19  shippingservice2-0   container.node-6.container_fs_writes_MB./dev/vda           61  0.000000e+00  0.000000e+00               0,                cmdb_id                           trace_name  total_count        p95        p5  incident_count
0   shippingservice2-0     trace.from_frontend2-0.row_count           30  12.000000  3.900000               0
1          adservice-0  trace.from_frontend-0.duration_mean           30   0.000029  0.000016               0
2          adservice-0   trace.from_frontend-0.duration_p95           30   0.000075  0.000018               0
3          adservice-0     trace.from_frontend-0.error_rate           30   0.000000  0.000000               0
4          adservice-0      trace.from_frontend-0.row_count           30   9.000000  6.000000               0
5          adservice-0  trace.from_frontend-1.duration_mean           30   0.000027  0.000017               0
6          adservice-0   trace.from_frontend-1.duration_p95           30   0.000069  0.000020               0
7          adservice-0     trace.from_frontend-1.error_rate           30   0.000000  0.000000               0
8          adservice-0      trace.from_frontend-1.row_count           30  10.000000  6.000000               0
9          adservice-0  trace.from_frontend-2.duration_mean           30   0.000020  0.000016               0
10         adservice-0   trace.from_frontend-2.duration_p95           30   0.000025  0.000016               0
11         adservice-0     trace.from_frontend-2.error_rate           30   0.000000  0.000000               0
12         adservice-0      trace.from_frontend-2.row_count           30   4.000000  1.000000               0
13         adservice-1  trace.from_frontend-0.duration_mean           30   0.000028  0.000016               0
14         adservice-1   trace.from_frontend-0.duration_p95           30   0.000056  0.000017               0
15         adservice-1     trace.from_frontend-0.error_rate           30   0.000000  0.000000               0
16   shippingservice-1  trace.from_frontend-1.duration_mean           30   0.000098  0.000067               0
17   shippingservice-1   trace.from_frontend-1.duration_p95           30   0.000144  0.000068               0
18   shippingservice-1     trace.from_frontend-1.error_rate           30   0.000000  0.000000               0
19   shippingservice-1      trace.from_frontend-1.row_count           30   4.000000  1.000000               0,                     cmdb_id         log_name  total_count    p95     p5  incident_count
0             cartservice-2    log.row_count           31  313.0  241.5               0
1             cartservice-2  log.error_count           31    0.0    0.0               0
2             cartservice-1    log.row_count           31  310.5  243.0               0
3             cartservice-1  log.error_count           31    0.0    0.0               0
4             cartservice-0    log.row_count           31  310.0  251.0               0
5             cartservice-0  log.error_count           31    0.0    0.0               0
6         currencyservice-0  log.error_count           31    0.0    0.0               0
7         currencyservice-0    log.row_count           31  301.5  194.0               0
8         currencyservice-2  log.error_count           31    0.0    0.0               0
9         currencyservice-2    log.row_count           31  304.0  196.0               0
10        currencyservice-1  log.error_count           31    0.0    0.0               0
11        currencyservice-1    log.row_count           31  304.0  193.5               0
12               frontend-0    log.row_count           31  642.5  496.5               0
13               frontend-0  log.error_count           31    0.0    0.0               0
14        shippingservice-1    log.row_count           31   47.5   17.5               0
15        shippingservice-1  log.error_count           31    0.0    0.0               0
16  recommendationservice-0    log.row_count           31   82.5   65.5               0
17  recommendationservice-0  log.error_count           31    0.0    0.0               0
18  recommendationservice-2  log.error_count           31    0.0    0.0               0
19  recommendationservice-2    log.row_count           31   81.0   64.0               0,          file  incident_rows
0  metric.csv              0
1   trace.csv              0
2     log.csv              0, 'Any rows in incident window? no. Counts per file: metric.csv=0, trace.csv=0, log.csv=0')```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import date

# Reuse existing DataFrames if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')
try:
    log
except NameError:
    log = pd.read_csv('log.csv')
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

tz = pytz.timezone('Asia/Shanghai')
target_date = date(2022, 3, 21)  # UTC+8 date to check

def ensure_ts_column(df):
    # Ensure df['ts'] exists and is timezone-aware (Asia/Shanghai)
    if 'ts' in df.columns and pd.api.types.is_datetime64_any_dtype(df['ts']):
        # if tz-naive, localize; if tz-aware but different, convert
        if df['ts'].dt.tz is None:
            df['ts'] = df['ts'].dt.tz_localize(tz)
        else:
            # convert to target tz to be safe
            df['ts'] = df['ts'].dt.tz_convert(tz)
    else:
        # create from integer 'timestamp' column if available
        if 'timestamp' in df.columns:
            df['ts'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
        else:
            # no timestamp info; create empty tz-aware series
            df['ts'] = pd.Series(pd.to_datetime([], unit='s')).dt.tz_localize(tz)
    return df

metric = ensure_ts_column(metric)
trace = ensure_ts_column(trace)
log = ensure_ts_column(log)
error_logs = ensure_ts_column(error_logs)

def summarize_file(df, name_col):
    # total rows
    total_rows = int(df.shape[0])
    # min/max ts
    if total_rows == 0:
        min_ts = None
        max_ts = None
    else:
        min_ts = df['ts'].min()
        max_ts = df['ts'].max()
    # rows matching target_date in UTC+8
    rows_on_date_mask = (df['ts'].dt.date == target_date)
    rows_on_date = int(rows_on_date_mask.sum())
    # prepare samples if any: up to earliest 5 distinct (cmdb_id, timestamp, name)
    if rows_on_date > 0:
        # select relevant columns and sort by ts asc
        sel = df.loc[rows_on_date_mask, :].copy()
        # build name column safe
        sel['_name'] = sel[name_col].astype(str) if name_col in sel.columns else pd.Series([""]*len(sel), index=sel.index)
        sel = sel.sort_values('ts', ascending=True)
        # Keep distinct by (cmdb_id, ts, _name)
        sel['ts_iso'] = sel['ts'].astype(str)
        sel = sel.drop_duplicates(subset=['cmdb_id','ts_iso','_name'])
        samples = sel.loc[:, ['cmdb_id','ts_iso','_name']].head(5)
        samples.columns = ['cmdb_id','timestamp','relevant_name']
    else:
        samples = pd.DataFrame(columns=['cmdb_id','timestamp','relevant_name'])
    summary = pd.DataFrame([{
        'total_rows': total_rows,
        'min_ts': min_ts,
        'max_ts': max_ts,
        'rows_on_2022-03-21': rows_on_date
    }])
    return summary, samples

# For metric: relevant name is kpi_name
metric_summary, metric_samples = summarize_file(metric, 'kpi_name')

# For trace: relevant name is trace_name
trace_summary, trace_samples = summarize_file(trace, 'trace_name')

# For log: relevant name is log_name
log_summary, log_samples = summarize_file(log, 'log_name')

# For error_logs: relevant name is message
error_summary, error_samples = summarize_file(error_logs, 'message')

# Compact overall check: whether any file had rows on that date
any_on_date = 'yes' if (int(metric_summary.at[0,'rows_on_2022-03-21']) +
                        int(trace_summary.at[0,'rows_on_2022-03-21']) +
                        int(log_summary.at[0,'rows_on_2022-03-21']) +
                        int(error_summary.at[0,'rows_on_2022-03-21'])) > 0 else 'no'
overall_line = f"Any telemetry on 2022-03-21 (UTC+8)? {any_on_date}. Counts per file: metric.csv={int(metric_summary.at[0,'rows_on_2022-03-21'])}, trace.csv={int(trace_summary.at[0,'rows_on_2022-03-21'])}, log.csv={int(log_summary.at[0,'rows_on_2022-03-21'])}, error_logs.csv={int(error_summary.at[0,'rows_on_2022-03-21'])}"

# Return compact results (stored for follow-ups)
metric_summary, metric_samples, trace_summary, trace_samples, log_summary, log_samples, error_summary, error_samples, overall_line

```
Out[3]:
```
metric_summary, trace_summary, log_summary, error_summary  # reuse existing summaries if present

def row_info(df):
    if df is None or df.empty:
        return 0, None, None, 0
    total = int(df.at[0, 'total_rows'])
    min_ts = df.at[0, 'min_ts']
    max_ts = df.at[0, 'max_ts']
    on_date = int(df.at[0, 'rows_on_2022-03-21'])
    # format timestamps as ISO strings if not None
    min_s = str(min_ts) if pd.notna(min_ts) else None
    max_s = str(max_ts) if pd.notna(max_ts) else None
    return total, min_s, max_s, on_date

m_tot, m_min, m_max, m_on = row_info(metric_summary)
t_tot, t_min, t_max, t_on = row_info(trace_summary)
l_tot, l_min, l_max, l_on = row_info(log_summary)
e_tot, e_min, e_max, e_on = row_info(error_summary)

any_on = 'yes' if (m_on + t_on + l_on + e_on) > 0 else 'no'

summary = (
    "Telemetry presence on 2022-03-21 (UTC+8):\n\n"
    f"metric.csv: total_rows={m_tot}, time_range={m_min} to {m_max}, rows_on_2022-03-21={m_on}\n"
    f"trace.csv:  total_rows={t_tot}, time_range={t_min} to {t_max}, rows_on_2022-03-21={t_on}\n"
    f"log.csv:    total_rows={l_tot}, time_range={l_min} to {l_max}, rows_on_2022-03-21={l_on}\n"
    f"error_logs.csv: total_rows={e_tot}, time_range={e_min} to {e_max}, rows_on_2022-03-21={e_on}\n\n"
    f"Overall: Any telemetry on 2022-03-21? {any_on}."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows                    min_ts                    max_ts  rows_on_2022-03-21
0      321472 2022-03-20 16:30:00+08:00 2022-03-20 17:30:00+08:00                   0, Empty DataFrame
Columns: [cmdb_id, timestamp, relevant_name]
Index: [],    total_rows                    min_ts                    max_ts  rows_on_2022-03-21
0       29584 2022-03-20 17:00:00+08:00 2022-03-20 17:29:00+08:00                   0, Empty DataFrame
Columns: [cmdb_id, timestamp, relevant_name]
Index: [],    total_rows                    min_ts                    max_ts  rows_on_2022-03-21
0        1750 2022-03-20 17:00:00+08:00 2022-03-20 17:30:00+08:00                   0, Empty DataFrame
Columns: [cmdb_id, timestamp, relevant_name]
Index: [],    total_rows min_ts max_ts  rows_on_2022-03-21
0           0   None   None                   0, Empty DataFrame
Columns: [cmdb_id, timestamp, relevant_name]
Index: [], 'Any telemetry on 2022-03-21 (UTC+8)? no. Counts per file: metric.csv=0, trace.csv=0, log.csv=0, error_logs.csv=0')```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing DataFrames if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')
try:
    log
except NameError:
    log = pd.read_csv('log.csv')
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

tz = pytz.timezone('Asia/Shanghai')
target_date_str = '2022-03-21'

def ensure_ts(df):
    if 'ts' in df.columns and pd.api.types.is_datetime64_any_dtype(df['ts']):
        if df['ts'].dt.tz is None:
            df['ts'] = df['ts'].dt.tz_localize(tz)
        else:
            df['ts'] = df['ts'].dt.tz_convert(tz)
    else:
        if 'timestamp' in df.columns and not df['timestamp'].isnull().all():
            df['ts'] = pd.to_datetime(df['timestamp'], unit='s').dt.tz_localize(tz)
        else:
            # create empty tz-aware datetime series
            df['ts'] = pd.Series(pd.to_datetime([], unit='s')).dt.tz_localize(tz)
    return df

metric = ensure_ts(metric)
trace = ensure_ts(trace)
log = ensure_ts(log)
error_logs = ensure_ts(error_logs)

def date_counts_table(df):
    if df.shape[0] == 0:
        tbl = pd.DataFrame(columns=['date','row_count'])
        has_target = False
    else:
        df = df.copy()
        df['date'] = df['ts'].dt.tz_convert(tz).dt.strftime('%Y-%m-%d')
        grp = df.groupby('date').size().reset_index(name='row_count').sort_values('date', ascending=True).reset_index(drop=True)
        tbl = grp.head(20)
        has_target = target_date_str in grp['date'].values
    return tbl, has_target

metric_date_counts, metric_has_2022_03_21 = date_counts_table(metric)
trace_date_counts, trace_has_2022_03_21 = date_counts_table(trace)
log_date_counts, log_has_2022_03_21 = date_counts_table(log)
error_date_counts, error_has_2022_03_21 = date_counts_table(error_logs)

has_flags = pd.DataFrame({
    'file': ['metric.csv','trace.csv','log.csv','error_logs.csv'],
    'has_2022-03-21': [metric_has_2022_03_21, trace_has_2022_03_21, log_has_2022_03_21, error_has_2022_03_21]
})

# Compact outputs (each is limited to top 20 rows). Stored for follow-ups.
metric_date_counts, trace_date_counts, log_date_counts, error_date_counts, has_flags

```
Out[4]:
```
summary = (
    "Per-date telemetry counts (dates in UTC+8):\n"
    "- metric.csv: 2022-03-20 -> 321,472 rows\n"
    "- trace.csv:  2022-03-20 -> 29,584 rows\n"
    "- log.csv:    2022-03-20 -> 1,750 rows\n"
    "- error_logs.csv: no rows (empty)\n\n"
    "Is 2022-03-21 present? No — none of the files contain rows dated 2022-03-21 (all False)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         date  row_count
0  2022-03-20     321472,          date  row_count
0  2022-03-20      29584,          date  row_count
0  2022-03-20       1750, Empty DataFrame
Columns: [date, row_count]
Index: [],              file  has_2022-03-21
0      metric.csv           False
1       trace.csv           False
2         log.csv           False
3  error_logs.csv           False)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse dataframes if already loaded in kernel; otherwise load from CSVs
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')
try:
    log
except NameError:
    log = pd.read_csv('log.csv')
try:
    error_logs
except NameError:
    error_logs = pd.read_csv('error_logs.csv')

tz_sh = pytz.timezone('Asia/Shanghai')  # UTC+8
tz_utc = pytz.UTC

# Helper to compute summary for one dataframe
def summarize_file_time(df, filename):
    out = {
        'file': filename,
        'raw_min_ts': pd.NA,
        'raw_max_ts': pd.NA,
        'min_ts_UTC': pd.NA,
        'max_ts_UTC': pd.NA,
        'min_ts_UTC+8': pd.NA,
        'max_ts_UTC+8': pd.NA,
        'count_in_window_UTC+8': 0,
        'count_in_window_UTC': 0
    }
    if df.shape[0] == 0 or 'timestamp' not in df.columns:
        return out
    # Coerce timestamp to numeric (epoch seconds); drop NA timestamps for min/max and counting
    ts_numeric = pd.to_numeric(df['timestamp'], errors='coerce')
    if ts_numeric.dropna().empty:
        return out
    raw_min = int(ts_numeric.min())
    raw_max = int(ts_numeric.max())
    out['raw_min_ts'] = raw_min
    out['raw_max_ts'] = raw_max
    # Convert raw_min/raw_max to UTC and UTC+8 datetimes
    min_utc_dt = pd.to_datetime(raw_min, unit='s', utc=True)
    max_utc_dt = pd.to_datetime(raw_max, unit='s', utc=True)
    out['min_ts_UTC'] = min_utc_dt.isoformat()
    out['max_ts_UTC'] = max_utc_dt.isoformat()
    out['min_ts_UTC+8'] = str(min_utc_dt.tz_convert(tz_sh))
    out['max_ts_UTC+8'] = str(max_utc_dt.tz_convert(tz_sh))
    # Prepare full timestamp series as tz-aware UTC datetimes for window checks
    ts_utc_series = pd.to_datetime(ts_numeric, unit='s', utc=True)
    # Window in UTC+8: 2022-03-21 01:00:00 to 2022-03-21 01:30:00 (inclusive)
    win_sh_start = tz_sh.localize(pd.to_datetime("2022-03-21 01:00:00"))
    win_sh_end   = tz_sh.localize(pd.to_datetime("2022-03-21 01:30:00"))
    # Convert UTC series to UTC+8 for checking
    ts_sh_series = ts_utc_series.dt.tz_convert(tz_sh)
    mask_sh = (ts_sh_series >= win_sh_start) & (ts_sh_series <= win_sh_end)
    out['count_in_window_UTC+8'] = int(mask_sh.sum())
    # Window in UTC: 2022-03-20 17:00:00 to 2022-03-20 17:30:00 (inclusive)
    win_utc_start = pd.to_datetime("2022-03-20 17:00:00").tz_localize(tz_utc)
    win_utc_end   = pd.to_datetime("2022-03-20 17:30:00").tz_localize(tz_utc)
    mask_utc = (ts_utc_series >= win_utc_start) & (ts_utc_series <= win_utc_end)
    out['count_in_window_UTC'] = int(mask_utc.sum())
    return out

rows = []
rows.append(summarize_file_time(metric, 'metric.csv'))
rows.append(summarize_file_time(trace, 'trace.csv'))
rows.append(summarize_file_time(log, 'log.csv'))
rows.append(summarize_file_time(error_logs, 'error_logs.csv'))

file_time_summary = pd.DataFrame(rows)[[
    'file',
    'raw_min_ts','raw_max_ts',
    'min_ts_UTC','max_ts_UTC',
    'min_ts_UTC+8','max_ts_UTC+8',
    'count_in_window_UTC+8','count_in_window_UTC'
]]

# Keep output compact
file_time_summary

```
Out[5]:
```
Summary (plain English):

- metric.csv
  - raw_min_ts = 1647793800, raw_max_ts = 1647797400
  - min_ts_UTC = 2022-03-20T16:30:00+00:00, max_ts_UTC = 2022-03-20T17:30:00+00:00
  - min_ts_UTC+8 = 2022-03-21 00:30:00+08:00, max_ts_UTC+8 = 2022-03-21 01:30:00+08:00
  - count_in_window_UTC+8 (2022-03-21 01:00:00–01:30:00+08): 163,372
  - count_in_window_UTC  (2022-03-20 17:00:00–17:30:00+00): 163,372

- trace.csv
  - raw_min_ts = 1647795600, raw_max_ts = 1647797340
  - min_ts_UTC = 2022-03-20T17:00:00+00:00, max_ts_UTC = 2022-03-20T17:29:00+00:00
  - min_ts_UTC+8 = 2022-03-21 01:00:00+08:00, max_ts_UTC+8 = 2022-03-21 01:29:00+08:00
  - count_in_window_UTC+8: 29,584
  - count_in_window_UTC: 29,584

- log.csv
  - raw_min_ts = 1647795600, raw_max_ts = 1647797400
  - min_ts_UTC = 2022-03-20T17:00:00+00:00, max_ts_UTC = 2022-03-20T17:30:00+00:00
  - min_ts_UTC+8 = 2022-03-21 01:00:00+08:00, max_ts_UTC+8 = 2022-03-21 01:30:00+08:00
  - count_in_window_UTC+8: 1,750
  - count_in_window_UTC: 1,750

- error_logs.csv
  - no timestamps / empty file (counts 0)

Conclusion: Yes — timezone interpretation explains the earlier “missing” telemetry. When epoch seconds are correctly interpreted as UTC and then converted to UTC+8, the metric/trace/log records do fall inside the incident window (2022-03-21 01:00–01:30 UTC+8 / 2022-03-20 17:00–17:30 UTC). Earlier checks that produced zero likely treated the epoch values as already in UTC+8 (i.e., incorrectly localized them), causing an 8-hour shift and missing matches.

The original code execution output of IPython Kernel is also provided below for reference:

file  raw_min_ts  raw_max_ts                 min_ts_UTC                 max_ts_UTC               min_ts_UTC+8               max_ts_UTC+8  count_in_window_UTC+8  count_in_window_UTC
0      metric.csv  1647793800  1647797400  2022-03-20T16:30:00+00:00  2022-03-20T17:30:00+00:00  2022-03-21 00:30:00+08:00  2022-03-21 01:30:00+08:00                 163372               163372
1       trace.csv  1647795600  1647797340  2022-03-20T17:00:00+00:00  2022-03-20T17:29:00+00:00  2022-03-21 01:00:00+08:00  2022-03-21 01:29:00+08:00                  29584                29584
2         log.csv  1647795600  1647797400  2022-03-20T17:00:00+00:00  2022-03-20T17:30:00+00:00  2022-03-21 01:00:00+08:00  2022-03-21 01:30:00+08:00                   1750                 1750
3  error_logs.csv        <NA>        <NA>                       <NA>                       <NA>                       <NA>                       <NA>                      0                    0```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse metric if present, otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure value numeric
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Define UTC+8 timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) Define allowed root-cause components (nodes, services, pods) from the problem statement
components = set([
    # nodes
    'node-1','node-2','node-3','node-4','node-5','node-6',
    # services (service-level names)
    'frontend','shippingservice','checkoutservice','currencyservice','adservice',
    'emailservice','cartservice','productcatalogservice','recommendationservice','paymentservice',
    # pods (replicas and -2/2 suffixes as provided)
    'frontend-0','frontend-1','frontend-2','frontend2-0',
    'shippingservice-0','shippingservice-1','shippingservice-2','shippingservice2-0',
    'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0',
    'currencyservice-0','currencyservice-1','currencyservice-2','currencyservice2-0',
    'adservice-0','adservice-1','adservice-2','adservice2-0',
    'emailservice-0','emailservice-1','emailservice-2','emailservice2-0',
    'cartservice-0','cartservice-1','cartservice-2','cartservice2-0',
    'productcatalogservice-0','productcatalogservice-1','productcatalogservice-2','productcatalogservice2-0',
    'recommendationservice-0','recommendationservice-1','recommendationservice-2','recommendationservice2-0',
    'paymentservice-0','paymentservice-1','paymentservice-2','paymentservice2-0'
])

# 1) Restrict metric to these components
metric_rc = metric[metric['cmdb_id'].isin(components)].copy()

# 2) Compute global P95 and P5 and total_count per (cmdb_id, kpi_name) using full series (before any filtering)
group_cols = ['cmdb_id','kpi_name']
# drop NA values for quantile calc but keep count including NaNs? We'll compute total_count as size (including NaN values)
total_counts = metric_rc.groupby(group_cols).size().reset_index(name='total_count')
# compute p95 and p5 on non-null values
quantiles = metric_rc.dropna(subset=['value']).groupby(group_cols)['value'].agg(
    p95=lambda x: x.quantile(0.95),
    p5 = lambda x: x.quantile(0.05)
).reset_index()
# thresholds stored in kernel memory
thresholds_df = total_counts.merge(quantiles, on=group_cols, how='left')

# 3) Filter metric rows to the incident window 2022-03-21 01:00:00 to 2022-03-21 01:30:00 (UTC+8)
# Convert epoch->UTC tz-aware then to UTC+8
metric_rc['ts_utc'] = pd.to_datetime(metric_rc['timestamp'], unit='s', utc=True)
metric_rc['ts_sh']  = metric_rc['ts_utc'].dt.tz_convert(tz)

win_start = tz.localize(pd.to_datetime("2022-03-21 01:00:00"))
win_end   = tz.localize(pd.to_datetime("2022-03-21 01:30:00"))

in_window_mask = (metric_rc['ts_sh'] >= win_start) & (metric_rc['ts_sh'] <= win_end)
metric_window = metric_rc.loc[in_window_mask].copy()

# 4) For each (cmdb_id, kpi_name) present in the incident window compute requested metrics
if metric_window.shape[0] == 0:
    # create empty DataFrame with required columns to keep kernel variables consistent
    incidents_agg = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','anomaly_count','earliest_anomaly_ts_UTC+8',
        'max_value_in_window','min_value_in_window','p95','p5','severity'
    ])
    affected_cmdbs = pd.DataFrame(columns=['cmdb_id'])
else:
    # group windowed data
    win_grp = metric_window.groupby(group_cols)
    win_stats = win_grp['value'].agg(
        total_points_in_window = 'size',
        anomaly_count_raw = lambda x: x.size  # placeholder, will compute after merging thresholds
    ).reset_index()
    # compute min/max in window
    minmax = win_grp['value'].agg(
        max_value_in_window = 'max',
        min_value_in_window = 'min'
    ).reset_index()
    # earliest anomaly ts will be computed after determining anomaly rows per-group
    win_merged = win_stats.merge(minmax, on=group_cols, how='left')
    # attach p95/p5 from thresholds (which were computed on full series)
    win_merged = win_merged.merge(thresholds_df, on=group_cols, how='left')
    # Now compute anomaly_count and earliest anomaly timestamp
    # Determine anomalies per row: value >= p95 OR value <= p5 (use group thresholds)
    # Merge thresholds into metric_window for row-level comparison
    metric_window_thresh = metric_window.merge(thresholds_df, on=group_cols, how='left')
    # define anomaly boolean per row (only when p95/p5 exist)
    def is_anom(row):
        v = row['value']
        p95 = row['p95']
        p5  = row['p5']
        if pd.isna(v) or (pd.isna(p95) and pd.isna(p5)):
            return False
        # treat comparisons carefully if p95/p5 NaN
        if (not pd.isna(p95)) and (v >= p95):
            return True
        if (not pd.isna(p5)) and (v <= p5):
            return True
        return False
    metric_window_thresh['is_anomaly'] = metric_window_thresh.apply(is_anom, axis=1)
    # aggregate anomaly counts and earliest anomaly ts per group
    anom_agg = metric_window_thresh[metric_window_thresh['is_anomaly']].groupby(group_cols).agg(
        anomaly_count = ('is_anomaly','sum'),
        earliest_anomaly_ts_UTC8 = ('ts_sh','min')
    ).reset_index()
    # merge anomaly info into win_merged
    incidents_agg = win_merged.merge(anom_agg, on=group_cols, how='left')
    incidents_agg['anomaly_count'] = incidents_agg['anomaly_count'].fillna(0).astype(int)
    incidents_agg['earliest_anomaly_ts_UTC+8'] = incidents_agg['earliest_anomaly_ts_UTC8'].astype(str).replace('NaT','')
    incidents_agg.drop(columns=['anomaly_count_raw','earliest_anomaly_ts_UTC8'], errors='ignore', inplace=True)
    # Ensure p95/p5 numeric
    incidents_agg['p95'] = pd.to_numeric(incidents_agg['p95'], errors='coerce')
    incidents_agg['p5']  = pd.to_numeric(incidents_agg['p5'], errors='coerce')
    # compute severity:
    # high_ratio = max_value_in_window / p95  (if p95>0)
    # low_ratio = p5 / min_value_in_window (if min_value_in_window>0)
    # severity = max(high_ratio (if >1), low_ratio (if >1)) else 0
    def compute_sev(row):
        maxv = row['max_value_in_window']
        minv = row['min_value_in_window']
        p95 = row['p95']
        p5  = row['p5']
        high_ratio = np.nan
        low_ratio = np.nan
        if pd.notna(p95) and p95 != 0 and pd.notna(maxv):
            high_ratio = maxv / p95
        if pd.notna(p5) and pd.notna(minv) and minv > 0:
            low_ratio = p5 / minv
        # Only consider >1 as severity (meaning exceeded threshold)
        candidates = []
        if pd.notna(high_ratio) and high_ratio > 1:
            candidates.append(high_ratio)
        if pd.notna(low_ratio) and low_ratio > 1:
            candidates.append(low_ratio)
        if len(candidates) == 0:
            return 0.0
        return float(max(candidates))
    incidents_agg['severity'] = incidents_agg.apply(compute_sev, axis=1)
    # final select and sort by anomaly_count desc then severity desc, limit top20
    incidents_agg = incidents_agg[[
        'cmdb_id','kpi_name','total_points_in_window','anomaly_count','earliest_anomaly_ts_UTC+8',
        'max_value_in_window','min_value_in_window','p95','p5','severity'
    ]]
    incidents_agg = incidents_agg.sort_values(['anomaly_count','severity'], ascending=[False,False]).reset_index(drop=True)
    incidents_agg = incidents_agg.head(20)
    # list of distinct cmdb_id with anomaly_count>0 (top 20)
    affected_cmdbs = incidents_agg.loc[incidents_agg['anomaly_count']>0, ['cmdb_id']].drop_duplicates().head(20).reset_index(drop=True)

# Store results in kernel variables for follow-up
thresholds_df  # global thresholds per (cmdb_id,kpi_name)
incidents_agg, affected_cmdbs

```
Out[6]:
```
summary = (
    "Summary of metric anomalies during 2022-03-21 01:00–01:30 (UTC+8):\n\n"
    "Overall:\n"
    "- Number of (cmdb_id, kpi_name) groups observed in-window with anomaly_count>0: 13 (from top results)\n"
    "- Distinct components with anomalies (up to 20): node-5, currencyservice-2, productcatalogservice2-0, productcatalogservice-2, node-6, productcatalogservice-0, shippingservice-0, checkoutservice-2, checkoutservice2-0, node-4, paymentservice-0, productcatalogservice-1, adservice-2\n\n"
    "Top anomaly examples (sorted by anomaly_count then severity):\n"
    "1) node-5 / system.io.rkb_s — anomaly_count=31, earliest_anomaly=2022-03-21 01:00:00+08:00, max=58.0, p95=2.0, severity≈29.0\n"
    "2) currencyservice-2 / container.node-6.container_cpu_cfs_throttled_seconds_total — anomaly_count=31, earliest_anomaly=2022-03-21 01:00:00+08:00, max≈0.12465, p95≈0.004908, severity≈25.40\n"
    "3) productcatalogservice2-0 / container.node-6.container_memory_failures.count — anomaly_count=31, earliest_anomaly=2022-03-21 01:00:00+08:00, max=148.5, p95=16.5, severity=9.0\n"
    "4) productcatalogservice2-0 / container.node-6.container_memory_failures.highest_bytes — anomaly_count=31, earliest_anomaly=2022-03-21 01:00:00+08:00, max=148.5, p95=16.5, severity=9.0\n"
    "5) productcatalogservice-2 / container.node-6.container_cpu_cfs_throttled_seconds_total — anomaly_count=31, earliest_anomaly=2022-03-21 01:00:00+08:00, max≈9.4241, p95≈1.3696, severity≈6.88\n\n"
    "Interpretation:\n"
    "- Multiple components (nodes and services/pods) show strong high-side anomalies (values >= p95) during the incident window. Node-level KPIs (e.g., node-5 IO and node-6 TCP retransmits) and container-level CPU throttling / memory-failure metrics stand out with the highest severities.\n"
    "- The list above is compact (top examples). Full aggregated tables and thresholds are stored in kernel variables 'incidents_agg' and 'thresholds_df' for follow-up analysis."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id                                           kpi_name  total_points_in_window  anomaly_count  earliest_anomaly_ts_UTC+8  max_value_in_window  min_value_in_window        p95   p5   severity
0                     node-5                                    system.io.rkb_s                      31             31  2022-03-21 01:00:00+08:00            58.000000                  0.0   2.000000  0.0  29.000000
1          currencyservice-2  container.node-6.container_cpu_cfs_throttled_s...                      31             31  2022-03-21 01:00:00+08:00             0.124648                  0.0   0.004908  0.0  25.396598
2   productcatalogservice2-0  container.node-6.container_memory_failures.con...                      31             31  2022-03-21 01:00:00+08:00           148.500000                  0.0  16.500000  0.0   9.000000
3   productcatalogservice2-0  container.node-6.container_memory_failures.hie...                      31             31  2022-03-21 01:00:00+08:00           148.500000                  0.0  16.500000  0.0   9.000000
4    productcatalogservice-2  container.node-6.container_cpu_cfs_throttled_s...                      31             31  2022-03-21 01:00:00+08:00             9.424077                  0.0   1.369564  0.0   6.881080
5                     node-6                        system.net.tcp.retrans_segs                      31             31  2022-03-21 01:00:00+08:00             0.130000                  0.0   0.020000  0.0   6.500000
6                     node-5                                      system.io.r_s                      31             31  2022-03-21 01:00:00+08:00             2.500000                  0.0   0.500000  0.0   5.000000
7    productcatalogservice-2   container.node-6.container_fs_writes_MB./dev/vda                      31             31  2022-03-21 01:00:00+08:00             0.042969                  0.0   0.013672  0.0   3.142857
8    productcatalogservice-2      container.node-6.container_fs_writes./dev/vda                      31             31  2022-03-21 01:00:00+08:00             4.000000                  0.0   1.500000  0.0   2.666667
9    productcatalogservice-2  container.node-6.container_cpu_cfs_throttled_p...                      31             31  2022-03-21 01:00:00+08:00            38.500000                  0.0  16.500000  0.0   2.333333
10   productcatalogservice-0  mesh.destination.recommendationservice2.produc...                      31             31  2022-03-21 01:00:00+08:00             0.001050                  0.0   0.000525  0.0   2.000000
11         shippingservice-0  mesh.destination.checkoutservice.shippingservi...                      31             31  2022-03-21 01:00:00+08:00             0.001050                  0.0   0.000525  0.0   2.000000
12         currencyservice-2  container.node-6.container_cpu_cfs_throttled_p...                      31             31  2022-03-21 01:00:00+08:00             1.000000                  0.0   0.500000  0.0   2.000000
13                    node-5                                  system.cpu.iowait                      31             31  2022-03-21 01:00:00+08:00             0.040000                  0.0   0.020000  0.0   2.000000
14         checkoutservice-2  mesh.source.checkoutservice.jaeger-collector.i...                      31             31  2022-03-21 01:00:00+08:00             0.001050                  0.0   0.000525  0.0   2.000000
15        checkoutservice2-0                 container.node-6.container_sockets                      31             31  2022-03-21 01:00:00+08:00             3.000000                  2.0   3.000000  3.0   1.500000
16                    node-4                        system.net.tcp.retrans_segs                      31             31  2022-03-21 01:00:00+08:00             0.030000                  0.0   0.020000  0.0   1.500000
17          paymentservice-0                 container.node-6.container_sockets                      31             31  2022-03-21 01:00:00+08:00             3.000000                  2.0   3.000000  2.5   1.250000
18   productcatalogservice-1                 container.node-6.container_sockets                      31             31  2022-03-21 01:00:00+08:00             5.000000                  4.0   4.000000  4.0   1.250000
19               adservice-2                 container.node-5.container_sockets                      31             31  2022-03-21 01:00:00+08:00             7.000000                  6.0   6.000000  6.0   1.166667,                      cmdb_id
0                     node-5
1          currencyservice-2
2   productcatalogservice2-0
3    productcatalogservice-2
4                     node-6
5    productcatalogservice-0
6          shippingservice-0
7          checkoutservice-2
8         checkoutservice2-0
9                     node-4
10          paymentservice-0
11   productcatalogservice-1
12               adservice-2)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing 'trace' DataFrame if present; otherwise load
try:
    trace
except NameError:
    trace = pd.read_csv('trace.csv')

# Ensure numeric values
trace['value'] = pd.to_numeric(trace['value'], errors='coerce')

# Timezones
tz_sh = pytz.timezone('Asia/Shanghai')  # UTC+8
tz_utc = pytz.UTC

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series (no time filtering)
group_cols = ['cmdb_id', 'trace_name']
# total count per group (including NaNs as rows)
trace_total_counts = trace.groupby(group_cols).size().reset_index(name='total_count')
# quantiles computed on non-null values
trace_quantiles = trace.dropna(subset=['value']).groupby(group_cols)['value'].agg(
    p95=lambda x: x.quantile(0.95),
    p5 = lambda x: x.quantile(0.05)
).reset_index()
# thresholds stored in kernel variable
trace_thresholds_df = trace_total_counts.merge(trace_quantiles, on=group_cols, how='left')

# 2) Filter trace rows to incident window 2022-03-21 01:00:00 to 2022-03-21 01:30:00 (UTC+8)
# create tz-aware UTC datetime then convert to UTC+8
trace['ts_utc'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
trace['ts_sh']  = trace['ts_utc'].dt.tz_convert(tz_sh)

win_start = tz_sh.localize(pd.to_datetime("2022-03-21 01:00:00"))
win_end   = tz_sh.localize(pd.to_datetime("2022-03-21 01:30:00"))

trace_window = trace[(trace['ts_sh'] >= win_start) & (trace['ts_sh'] <= win_end)].copy()

# 3) For each (cmdb_id, trace_name) present in the window compute requested stats
if trace_window.shape[0] == 0:
    trace_incidents_agg = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_points_in_window','anomaly_count','earliest_anomaly_ts_UTC+8',
        'max_value_in_window','min_value_in_window','p95','p5','severity'
    ])
    trace_affected_cmdbs = pd.DataFrame(columns=['cmdb_id','earliest_anomaly_ts_UTC+8'])
else:
    # window group stats
    win_grp = trace_window.groupby(group_cols)
    win_stats = win_grp['value'].agg(
        total_points_in_window = 'size',
        max_value_in_window = 'max',
        min_value_in_window = 'min'
    ).reset_index()
    # merge global thresholds (p95/p5) into window groups
    win_merged = win_stats.merge(trace_thresholds_df, on=group_cols, how='left')
    # merge thresholds into row level window for anomaly detection
    trace_win_thresh = trace_window.merge(trace_thresholds_df, on=group_cols, how='left')
    # define anomaly boolean per row
    def row_is_anom(r):
        v = r['value']
        p95 = r['p95']
        p5  = r['p5']
        if pd.isna(v):
            return False
        # high-side
        if pd.notna(p95) and v >= p95:
            return True
        # low-side
        if pd.notna(p5) and v <= p5:
            return True
        return False
    trace_win_thresh['is_anomaly'] = trace_win_thresh.apply(row_is_anom, axis=1)
    # aggregate anomaly counts and earliest anomaly timestamp per (cmdb_id, trace_name)
    anom_agg = trace_win_thresh[trace_win_thresh['is_anomaly']].groupby(group_cols).agg(
        anomaly_count = ('is_anomaly','sum'),
        earliest_anomaly_ts = ('ts_sh','min')
    ).reset_index()
    # merge anomaly info into win_merged
    trace_incidents_agg = win_merged.merge(anom_agg, on=group_cols, how='left')
    trace_incidents_agg['anomaly_count'] = trace_incidents_agg['anomaly_count'].fillna(0).astype(int)
    # format earliest timestamp strings
    trace_incidents_agg['earliest_anomaly_ts_UTC+8'] = trace_incidents_agg['earliest_anomaly_ts'].astype(str).replace('NaT','')
    trace_incidents_agg.drop(columns=['earliest_anomaly_ts'], inplace=True, errors='ignore')
    # ensure p95/p5 numeric
    trace_incidents_agg['p95'] = pd.to_numeric(trace_incidents_agg['p95'], errors='coerce')
    trace_incidents_agg['p5']  = pd.to_numeric(trace_incidents_agg['p5'], errors='coerce')
    # compute severity per group
    def compute_severity(r):
        maxv = r['max_value_in_window']
        minv = r['min_value_in_window']
        p95 = r['p95']
        p5  = r['p5']
        candidates = []
        if pd.notna(p95) and p95 != 0 and pd.notna(maxv):
            high_ratio = maxv / p95
            if high_ratio > 1:
                candidates.append(high_ratio)
        if pd.notna(p5) and pd.notna(minv) and minv > 0:
            low_ratio = p5 / minv
            if low_ratio > 1:
                candidates.append(low_ratio)
        return float(max(candidates)) if candidates else 0.0
    trace_incidents_agg['severity'] = trace_incidents_agg.apply(compute_severity, axis=1)
    # select columns and sort by anomaly_count desc then severity desc, limit top20
    trace_incidents_agg = trace_incidents_agg[[
        'cmdb_id','trace_name','total_points_in_window','anomaly_count','earliest_anomaly_ts_UTC+8',
        'max_value_in_window','min_value_in_window','p95','p5','severity'
    ]]
    trace_incidents_agg = trace_incidents_agg.sort_values(['anomaly_count','severity'], ascending=[False,False]).reset_index(drop=True).head(20)
    # 5) build distinct cmdb_id list with earliest anomaly timestamp among its trace metrics
    anom_rows = trace_incidents_agg[trace_incidents_agg['anomaly_count']>0].copy()
    if anom_rows.shape[0] == 0:
        trace_affected_cmdbs = pd.DataFrame(columns=['cmdb_id','earliest_anomaly_ts_UTC+8'])
    else:
        trace_affected_cmdbs = anom_rows.groupby('cmdb_id', as_index=False).agg(
            earliest_anomaly_ts_UTC_plus8 = ('earliest_anomaly_ts_UTC+8','min')
        ).rename(columns={'earliest_anomaly_ts_UTC_plus8':'earliest_anomaly_ts_UTC+8'}).sort_values('earliest_anomaly_ts_UTC+8').head(20).reset_index(drop=True)

# Store thresholds in kernel variable: trace_thresholds_df
# Output compact tables for follow-up
trace_thresholds_df.head(20), trace_incidents_agg, trace_affected_cmdbs

```
Out[7]:
```
Summary (plain English):

- Thresholds: global P95 and P5 were computed per (cmdb_id, trace_name) for the full trace series and stored in kernel variable trace_thresholds_df.

- Windowed anomalies (2022-03-21 01:00–01:30 UTC+8):
  - Many trace groups in the incident window show anomalies: the top groups all have total_points_in_window = 30 and anomaly_count = 30 (i.e., every point in-window is flagged by >=P95 or <=P5).
  - Earliest anomalies for the top groups occur at the start of the window: 2022-03-21 01:00:00+08:00.
  - Example top entries (sorted by anomaly_count then severity):
    1) frontend-0 / trace.to_shippingservice-1.row_count — total=30, anomaly_count=30, earliest=2022-03-21 01:00:00+08:00, max=3, min=1, p95=3, p5=2, severity=2.0
    2) shippingservice-1 / trace.from_frontend-0.row_count — total=30, anomaly_count=30, earliest=2022-03-21 01:00:00+08:00, max=3, min=1, p95=3, p5=2, severity=2.0
    3) cartservice-2 / trace.self.duration_mean — total=30, anomaly_count=30, earliest=2022-03-21 01:00:00+08:00, max≈3.4e-05, p95≈1.8e-05, severity≈1.87
    4) Many adservice.* and cartservice.* trace.error_rate metrics are flagged (anomaly_count=30) but have p95/p5 = 0 and values = 0, so computed severity is 0 (they meet threshold equality but reflect zero-valued error-rate series).
  - In short: multiple services and trace dimensions show full-window anomalies, with earliest occurrences at the window start.

- Distinct components with trace anomalies (top 20, ordered by earliest anomaly): 
  adservice-0, adservice-1, adservice-2, adservice2-0, cartservice-0, cartservice-1, cartservice-2, frontend-0, shippingservice-1
  (these are the components that had anomaly_count>0; earliest anomaly timestamps are 2022-03-21 01:00:00+08:00).

- Outputs saved for follow-up:
  - trace_thresholds_df (global p95/p5 per group)
  - trace_incidents_agg (windowed per-group stats and severity)
  - trace_affected_cmdbs (distinct cmdb_id list with earliest anomaly ts)

If you want, I can: (a) expand the top-20 table for inspection, (b) cross-correlate these trace anomalies with the metric or log anomalies found earlier, or (c) export the specific anomaly rows for one or more components for deeper investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                           trace_name  total_count        p95        p5
0   adservice-0  trace.from_frontend-0.duration_mean           30   0.000029  0.000016
1   adservice-0   trace.from_frontend-0.duration_p95           30   0.000075  0.000018
2   adservice-0     trace.from_frontend-0.error_rate           30   0.000000  0.000000
3   adservice-0      trace.from_frontend-0.row_count           30   9.000000  6.000000
4   adservice-0  trace.from_frontend-1.duration_mean           30   0.000027  0.000017
5   adservice-0   trace.from_frontend-1.duration_p95           30   0.000069  0.000020
6   adservice-0     trace.from_frontend-1.error_rate           30   0.000000  0.000000
7   adservice-0      trace.from_frontend-1.row_count           30  10.000000  6.000000
8   adservice-0  trace.from_frontend-2.duration_mean           30   0.000020  0.000016
9   adservice-0   trace.from_frontend-2.duration_p95           30   0.000025  0.000016
10  adservice-0     trace.from_frontend-2.error_rate           30   0.000000  0.000000
11  adservice-0      trace.from_frontend-2.row_count           30   4.000000  1.000000
12  adservice-1  trace.from_frontend-0.duration_mean           30   0.000028  0.000016
13  adservice-1   trace.from_frontend-0.duration_p95           30   0.000056  0.000017
14  adservice-1     trace.from_frontend-0.error_rate           30   0.000000  0.000000
15  adservice-1      trace.from_frontend-0.row_count           30   9.000000  6.000000
16  adservice-1  trace.from_frontend-1.duration_mean           30   0.000028  0.000016
17  adservice-1   trace.from_frontend-1.duration_p95           30   0.000072  0.000017
18  adservice-1     trace.from_frontend-1.error_rate           30   0.000000  0.000000
19  adservice-1      trace.from_frontend-1.row_count           30  10.000000  6.000000,               cmdb_id                            trace_name  total_points_in_window  anomaly_count  earliest_anomaly_ts_UTC+8  max_value_in_window  min_value_in_window       p95   p5  severity
0          frontend-0  trace.to_shippingservice-1.row_count                      30             30  2022-03-21 01:00:00+08:00             3.000000                  1.0  3.000000  2.0  2.000000
1   shippingservice-1       trace.from_frontend-0.row_count                      30             30  2022-03-21 01:00:00+08:00             3.000000                  1.0  3.000000  2.0  2.000000
2       cartservice-2              trace.self.duration_mean                      30             30  2022-03-21 01:00:00+08:00             0.000034                  0.0  0.000018  0.0  1.873278
3         adservice-0      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
4         adservice-0      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
5         adservice-0      trace.from_frontend-2.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
6         adservice-1      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
7         adservice-1      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
8         adservice-1      trace.from_frontend-2.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
9         adservice-2      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
10        adservice-2      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
11        adservice-2      trace.from_frontend-2.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
12       adservice2-0     trace.from_frontend2-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
13      cartservice-0      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
14      cartservice-0      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
15      cartservice-0      trace.from_frontend-2.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
16      cartservice-0               trace.self.duration_p95                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
17      cartservice-0                 trace.self.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
18      cartservice-1      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000
19      cartservice-1      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000                  0.0  0.000000  0.0  0.000000,              cmdb_id  earliest_anomaly_ts_UTC+8
0        adservice-0  2022-03-21 01:00:00+08:00
1        adservice-1  2022-03-21 01:00:00+08:00
2        adservice-2  2022-03-21 01:00:00+08:00
3       adservice2-0  2022-03-21 01:00:00+08:00
4      cartservice-0  2022-03-21 01:00:00+08:00
5      cartservice-1  2022-03-21 01:00:00+08:00
6      cartservice-2  2022-03-21 01:00:00+08:00
7         frontend-0  2022-03-21 01:00:00+08:00
8  shippingservice-1  2022-03-21 01:00:00+08:00)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse existing kernel variables: incidents_agg (metric), trace_incidents_agg (trace)
# If not present, attempt to load them as empty DataFrames to avoid errors
try:
    incidents_agg
except NameError:
    incidents_agg = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','total_points_in_window','anomaly_count','earliest_anomaly_ts_UTC+8',
        'max_value_in_window','min_value_in_window','p95','p5','severity'
    ])

try:
    trace_incidents_agg
except NameError:
    trace_incidents_agg = pd.DataFrame(columns=[
        'cmdb_id','trace_name','total_points_in_window','anomaly_count','earliest_anomaly_ts_UTC+8',
        'max_value_in_window','min_value_in_window','p95','p5','severity'
    ])

tz_sh = pytz.timezone('Asia/Shanghai')

# 1) Identify cmdb_ids with anomaly_count>0 in either metrics or traces
metric_anoms = incidents_agg[incidents_agg['anomaly_count']>0].copy()
trace_anoms  = trace_incidents_agg[trace_incidents_agg['anomaly_count']>0].copy()

metric_cmdbs = set(metric_anoms['cmdb_id'].unique())
trace_cmdbs  = set(trace_anoms['cmdb_id'].unique())
all_cmdbs = sorted(metric_cmdbs.union(trace_cmdbs))

# 2) Helper to parse earliest_anomaly_ts strings to tz-aware datetimes (UTC+8)
def parse_ts_series(s):
    # s may contain empty strings or NaN
    return pd.to_datetime(s, errors='coerce').dt.tz_convert(tz_sh) if not s.empty else pd.Series([], dtype='datetime64[ns, Asia/Shanghai]')

# Prepare per-cmdb_id aggregations
rows = []
# service names list (from provided domain knowledge)
service_names = set([
    'frontend','shippingservice','checkoutservice','currencyservice','adservice','emailservice',
    'cartservice','productcatalogservice','recommendationservice','paymentservice'
])
nodes_set = set([f'node-{i}' for i in range(1,7)])

for cm in all_cmdbs:
    # level determination
    if cm in nodes_set:
        level = 'node'
    elif cm in service_names:
        level = 'service'
    elif '-' in cm:
        # likely a pod (e.g., frontend-0, checkoutservice-1)
        level = 'pod'
    else:
        level = 'service'
    # metric-derived values
    m_df = metric_anoms[metric_anoms['cmdb_id']==cm]
    metric_anomaly_groups = int(m_df.shape[0])  # distinct kpi_name groups with anomaly_count>0 per incidents_agg rows
    # but incidents_agg has one row per (cmdb_id,kpi_name) so count rows
    max_metric_severity = float(m_df['severity'].max()) if not m_df.empty else 0.0
    # earliest anomaly ts from metrics (parse)
    if not m_df.empty:
        m_ts = pd.to_datetime(m_df['earliest_anomaly_ts_UTC+8'], errors='coerce')
        # ensure tz-awareness: if parsed tz-naive, localize to UTC+8
        m_ts = m_ts.apply(lambda x: pd.NaT if pd.isna(x) else (x.tz_localize(tz_sh) if x.tzinfo is None else x.tz_convert(tz_sh)))
        m_earliest = m_ts.min()
    else:
        m_earliest = pd.NaT
    # trace-derived values
    t_df = trace_anoms[trace_anoms['cmdb_id']==cm]
    trace_anomaly_groups = int(t_df.shape[0])
    max_trace_severity = float(t_df['severity'].max()) if not t_df.empty else 0.0
    if not t_df.empty:
        t_ts = pd.to_datetime(t_df['earliest_anomaly_ts_UTC+8'], errors='coerce')
        t_ts = t_ts.apply(lambda x: pd.NaT if pd.isna(x) else (x.tz_localize(tz_sh) if x.tzinfo is None else x.tz_convert(tz_sh)))
        t_earliest = t_ts.min()
    else:
        t_earliest = pd.NaT
    # combined earliest
    candidates = [ts for ts in [m_earliest, t_earliest] if pd.notna(ts)]
    if len(candidates) == 0:
        earliest = pd.NaT
    else:
        earliest = min(candidates)
    # format earliest as ISO string with tz
    earliest_str = str(earliest) if pd.notna(earliest) else ''
    rows.append({
        'cmdb_id': cm,
        'level': level,
        'earliest_anomaly_ts_UTC+8': earliest_str,
        'metric_anomaly_groups': metric_anomaly_groups,
        'trace_anomaly_groups': trace_anomaly_groups,
        'max_metric_severity': max_metric_severity,
        'max_trace_severity': max_trace_severity
    })

combined_candidates = pd.DataFrame(rows)

# Sort by earliest_anomaly_ts ascending (NaT last), then by max_metric_severity descending
# Convert earliest string back to datetime for sorting
combined_candidates['_earliest_dt'] = pd.to_datetime(combined_candidates['earliest_anomaly_ts_UTC+8'], errors='coerce')
combined_candidates = combined_candidates.sort_values(by=['_earliest_dt','max_metric_severity'], ascending=[True, False]).reset_index(drop=True)
combined_candidates.drop(columns=['_earliest_dt'], inplace=True)

# Limit to top 20 rows for compactness
combined_candidates = combined_candidates.head(20)

# Short summary line: top 5 candidate cmdb_id with level and earliest_anomaly_ts_UTC+8
top5 = combined_candidates.head(5)
summary_items = top5.apply(lambda r: f"{r['cmdb_id']}({r['level']}, {r['earliest_anomaly_ts_UTC+8']})", axis=1).tolist()
summary_line = "Top candidates: " + ", ".join(summary_items) if summary_items else "Top candidates: none"

# Return compact table and summary_line (stored 'combined_candidates' for follow-up)
combined_candidates, summary_line

```
Out[8]:
```
# Compose a concise plain-English summary based on combined_candidates (already in kernel)
try:
    combined_candidates
except NameError:
    combined_candidates = None

if combined_candidates is None or combined_candidates.shape[0] == 0:
    summary = "No combined anomaly candidates found in kernel variable 'combined_candidates'."
else:
    topN = combined_candidates.head(5)
    lines = []
    lines.append("Combined anomaly candidates (from metrics + traces) — top findings:")
    lines.append(f"- Total distinct candidates: {len(combined_candidates)} (showing top {min(5,len(combined_candidates))}):")
    for _, r in topN.iterrows():
        lines.append(
            f"  • {r['cmdb_id']} ({r['level']}): earliest={r['earliest_anomaly_ts_UTC+8']}, "
            f"metric_groups={int(r['metric_anomaly_groups'])}, trace_groups={int(r['trace_anomaly_groups'])}, "
            f"max_metric_severity={r['max_metric_severity']:.3f}, max_trace_severity={r['max_trace_severity']:.3f}"
        )
    # High-level interpretation
    lines.append("")
    lines.append("Interpretation:")
    lines.append("- Node-level and certain pod-level components show the strongest metric anomalies (node-5 and node-6 stand out).")
    lines.append("- Notable pods/services with large metric severities: currencyservice-2, productcatalogservice2-0, productcatalogservice-2.")
    lines.append("- Several pods (e.g., adservice-*, cartservice-*) show trace anomalies (trace-group counts > 0) with earliest anomalies at the same window start.")
    lines.append("- Earliest anomalies for all top candidates occur at 2022-03-21 01:00:00+08:00, indicating a synchronized onset across metrics and traces.")
    lines.append("")
    lines.append("Notes:")
    lines.append("- The combined table is stored in kernel variable 'combined_candidates' for follow-up (log correlation, drill-down).")
    summary = "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id level  earliest_anomaly_ts_UTC+8  metric_anomaly_groups  trace_anomaly_groups  max_metric_severity  max_trace_severity
0                     node-5  node  2022-03-21 01:00:00+08:00                      3                     0            29.000000            0.000000
1          currencyservice-2   pod  2022-03-21 01:00:00+08:00                      2                     0            25.396598            0.000000
2   productcatalogservice2-0   pod  2022-03-21 01:00:00+08:00                      2                     0             9.000000            0.000000
3    productcatalogservice-2   pod  2022-03-21 01:00:00+08:00                      4                     0             6.881080            0.000000
4                     node-6  node  2022-03-21 01:00:00+08:00                      1                     0             6.500000            0.000000
5    productcatalogservice-0   pod  2022-03-21 01:00:00+08:00                      1                     0             2.000000            0.000000
6          shippingservice-0   pod  2022-03-21 01:00:00+08:00                      1                     0             2.000000            0.000000
7          checkoutservice-2   pod  2022-03-21 01:00:00+08:00                      1                     0             2.000000            0.000000
8         checkoutservice2-0   pod  2022-03-21 01:00:00+08:00                      1                     0             1.500000            0.000000
9                     node-4  node  2022-03-21 01:00:00+08:00                      1                     0             1.500000            0.000000
10          paymentservice-0   pod  2022-03-21 01:00:00+08:00                      1                     0             1.250000            0.000000
11   productcatalogservice-1   pod  2022-03-21 01:00:00+08:00                      1                     0             1.250000            0.000000
12               adservice-2   pod  2022-03-21 01:00:00+08:00                      1                     3             1.166667            0.000000
13               adservice-0   pod  2022-03-21 01:00:00+08:00                      0                     3             0.000000            0.000000
14               adservice-1   pod  2022-03-21 01:00:00+08:00                      0                     3             0.000000            0.000000
15              adservice2-0   pod  2022-03-21 01:00:00+08:00                      0                     1             0.000000            0.000000
16             cartservice-0   pod  2022-03-21 01:00:00+08:00                      0                     5             0.000000            0.000000
17             cartservice-1   pod  2022-03-21 01:00:00+08:00                      0                     2             0.000000            0.000000
18             cartservice-2   pod  2022-03-21 01:00:00+08:00                      0                     1             0.000000            1.873278
19                frontend-0   pod  2022-03-21 01:00:00+08:00                      0                     1             0.000000            2.000000, 'Top candidates: node-5(node, 2022-03-21 01:00:00+08:00), currencyservice-2(pod, 2022-03-21 01:00:00+08:00), productcatalogservice2-0(pod, 2022-03-21 01:00:00+08:00), productcatalogservice-2(pod, 2022-03-21 01:00:00+08:00), node-6(node, 2022-03-21 01:00:00+08:00)')```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing metric DataFrame if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv('metric.csv')

# Ensure numeric value column
metric['value'] = pd.to_numeric(metric['value'], errors='coerce')

# Timezone
tz = pytz.timezone('Asia/Shanghai')

# Ensure raw global thresholds for metric (per (cmdb_id,kpi_name)) exist in kernel variable 'thresholds_df'
# If not present, compute now using entire metric series (rule 9).
try:
    thresholds_df
except NameError:
    group_cols = ['cmdb_id','kpi_name']
    total_counts = metric.groupby(group_cols).size().reset_index(name='total_count')
    quantiles = metric.dropna(subset=['value']).groupby(group_cols)['value'].agg(
        p95=lambda x: x.quantile(0.95),
        p5 = lambda x: x.quantile(0.05)
    ).reset_index()
    thresholds_df = total_counts.merge(quantiles, on=group_cols, how='left')

# 1) Filter metric rows to incident window 2022-03-21 01:00:00 to 2022-03-21 01:30:00 (UTC+8)
# Create tz-aware UTC datetimes and convert to UTC+8
metric['ts_utc'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
metric['ts_sh']  = metric['ts_utc'].dt.tz_convert(tz)

win_start = tz.localize(pd.to_datetime("2022-03-21 01:00:00"))
win_end   = tz.localize(pd.to_datetime("2022-03-21 01:30:00"))

metric_window = metric[(metric['ts_sh'] >= win_start) & (metric['ts_sh'] <= win_end)].copy()

# 2) Mark rows as anomalous if value >= global P95 OR value <= global P5
# Merge thresholds into metric_window for comparison
metric_window = metric_window.merge(thresholds_df, on=['cmdb_id','kpi_name'], how='left')

# Define anomaly flag (handle NaNs)
def is_anomaly_row(r):
    v = r['value']
    p95 = r['p95']
    p5  = r['p5']
    if pd.isna(v):
        return False
    if pd.notna(p95) and v >= p95:
        return True
    if pd.notna(p5) and v <= p5:
        return True
    return False

metric_window['is_anomaly'] = metric_window.apply(is_anomaly_row, axis=1)

# Keep only anomalous rows for fault segmentation
anom_rows = metric_window[metric_window['is_anomaly']].copy()

# 3) Group anomalous rows by (cmdb_id,kpi_name) and identify consecutive-minute segments
# Prepare for segmentation: sort by ts_utc (or ts_sh)
if anom_rows.shape[0] == 0:
    # Empty results
    metric_fault_segments = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','fault_start_ts_UTC+8','fault_end_ts_UTC+8','duration_minutes',
        'points_in_segment','max_value_in_segment','min_value_in_segment','p95','p5','severity_metric'
    ])
else:
    anom_rows = anom_rows.sort_values(['cmdb_id','kpi_name','ts_utc']).reset_index(drop=True)
    # compute time diffs in seconds within each group
    anom_rows['prev_ts'] = anom_rows.groupby(['cmdb_id','kpi_name'])['ts_utc'].shift(1)
    anom_rows['diff_s'] = (anom_rows['ts_utc'] - anom_rows['prev_ts']).dt.total_seconds()
    # A new segment starts when diff_s != 60 or prev_ts is NaT
    anom_rows['new_segment'] = ((anom_rows['diff_s'] != 60) | anom_rows['diff_s'].isna()).astype(int)
    # assign segment id per group by cumulative sum of new_segment
    anom_rows['segment_id'] = anom_rows.groupby(['cmdb_id','kpi_name'])['new_segment'].cumsum()
    # Build segment key
    anom_rows['seg_key'] = anom_rows['cmdb_id'] + '||' + anom_rows['kpi_name'] + '||' + anom_rows['segment_id'].astype(str)
    # Aggregate per segment
    seg_grp = anom_rows.groupby(['cmdb_id','kpi_name','segment_id'], as_index=False)
    segs = seg_grp.agg(
        fault_start_ts_UTC_plus8 = ('ts_sh','min'),
        fault_end_ts_UTC_plus8   = ('ts_sh','max'),
        points_in_segment        = ('value','size'),
        max_value_in_segment     = ('value','max'),
        min_value_in_segment     = ('value','min'),
        p95                      = ('p95','first'),
        p5                       = ('p5','first')
    )
    # duration_minutes inclusive
    segs['duration_minutes'] = ((segs['fault_end_ts_UTC_plus8'] - segs['fault_start_ts_UTC_plus8']).dt.total_seconds() / 60).astype(int) + 1
    # compute severity_metric per instructions
    small_eps = 1e-9
    def seg_severity(row):
        maxv = row['max_value_in_segment']
        minv = row['min_value_in_segment']
        p95 = row['p95']
        p5  = row['p5']
        high_sev = 0.0
        low_sev = 0.0
        # high-side
        if pd.notna(p95):
            if p95 == 0:
                # per instruction: set severity_metric = max_value_in_segment
                high_sev = float(maxv) if pd.notna(maxv) else 0.0
            else:
                if pd.notna(maxv):
                    high_sev = float(maxv) / float(p95) if p95 != 0 else float(maxv)
        # low-side
        if pd.notna(p5) and pd.notna(minv):
            denom = float(minv) if float(minv) > 0 else small_eps
            low_sev = float(p5) / denom if denom != 0 else float(p5) / small_eps
        # Return the larger as overall severity
        return float(max(high_sev if not pd.isna(high_sev) else 0.0, low_sev if not pd.isna(low_sev) else 0.0))
    segs['severity_metric'] = segs.apply(seg_severity, axis=1)
    # Format timestamp columns as ISO strings
    segs['fault_start_ts_UTC+8'] = segs['fault_start_ts_UTC_plus8'].astype(str)
    segs['fault_end_ts_UTC+8']   = segs['fault_end_ts_UTC_plus8'].astype(str)
    metric_fault_segments = segs[[
        'cmdb_id','kpi_name','fault_start_ts_UTC+8','fault_end_ts_UTC+8','duration_minutes',
        'points_in_segment','max_value_in_segment','min_value_in_segment','p95','p5','severity_metric'
    ]].copy()
    # Sort by severity desc then duration desc
    metric_fault_segments = metric_fault_segments.sort_values(['severity_metric','duration_minutes'], ascending=[False,False]).reset_index(drop=True)

# 4) Return compact table: top 20 for display, but keep full table in kernel variable 'metric_fault_segments'
# metric_fault_segments variable already holds full table

# 5) Per-cmdb_id summary table (top 20)
# Determine level mapping
service_names = set([
    'frontend','shippingservice','checkoutservice','currencyservice','adservice','emailservice',
    'cartservice','productcatalogservice','recommendationservice','paymentservice'
])
nodes_set = set([f'node-{i}' for i in range(1,7)])

if metric_fault_segments.shape[0] == 0:
    per_cmdb_summary = pd.DataFrame(columns=[
        'cmdb_id','level','total_fault_segments_in_window','earliest_fault_start_UTC+8','max_severity_among_its_faults'
    ])
else:
    per = metric_fault_segments.groupby('cmdb_id').agg(
        total_fault_segments_in_window = ('kpi_name','size'),
        earliest_fault_start_UTC8 = ('fault_start_ts_UTC+8','min'),
        max_severity_among_its_faults = ('severity_metric','max')
    ).reset_index()
    # determine level
    def determine_level(cm):
        if cm in nodes_set:
            return 'node'
        if cm in service_names:
            return 'service'
        if '-' in cm:
            return 'pod'
        return 'service'
    per['level'] = per['cmdb_id'].apply(determine_level)
    per = per[['cmdb_id','level','total_fault_segments_in_window','earliest_fault_start_UTC8','max_severity_among_its_faults']]
    per_cmdb_summary = per.sort_values(['earliest_fault_start_UTC8','max_severity_among_its_faults'], ascending=[True,False]).reset_index(drop=True).head(20)

# Keep full fault segments in kernel variable name required
metric_fault_segments_full = metric_fault_segments.copy()  # full table stored
metric_fault_segments_head20 = metric_fault_segments_full.head(20)

# Compact outputs for display: metric_fault_segments_head20, per_cmdb_summary
metric_fault_segments_head20, per_cmdb_summary, metric_fault_segments_full

```
Out[9]:
```
Summary (plain English):

- Overall result
  - Total fault segments detected in the incident window: 9,636 (full table stored in kernel variable metric_fault_segments_full).
  - I extracted contiguous “fault” segments (consecutive anomalous minutes) per (cmdb_id, kpi_name) and computed duration, points, min/max, global p95/p5, and a severity_metric.

- Top fault segments (highest severity examples, top rows)
  1. productcatalogservice-2 / mesh.destination.checkoutservice.productcatalog...  
     - fault: 2022-03-21 01:29 → 01:30 (+08), duration 2 min, points=2  
     - max=6525.0, p95=5075.0, severity ≈ 1.45e+12 (highest)
  2. emailservice-2 / mesh.source.emailservice.jaeger-collector...  
     - fault: 2022-03-21 01:10 (single point)  
     - max=0.0, p95=3900.0, severity ≈ 9.75e+11
  3. paymentservice-0 / mesh.destination.checkoutservice.paymentservice...  
     - fault: 2022-03-21 01:03 → 01:06 (+08), duration 4 min, points=4  
     - max=3375.0, p95=3375.0, severity ≈ 6.75e+11
  4. productcatalogservice-2 / another mesh.destination… metric (similar)  
     - fault: 01:29 → 01:30, severity ≈ 6.55e+11
  5. emailservice-2 / mesh.destination.checkoutservice.emailservice...  
     - fault: 01:10 → 01:11, severity ≈ 6.25e+11

  - (Many other high-severity faults follow; the head-20 table was produced and the full ranked list is in metric_fault_segments_full.)

- Per-component (per cmdb_id) summary (top entries)
  - productcatalogservice-2 (pod) — total_fault_segments_in_window = 275; earliest fault = 2022-03-21 01:00:00+08; max severity among its faults ≈ 1.45e+12
  - emailservice-2 (pod) — 154 segments; earliest 2022-03-21 01:00:00+08; max severity ≈ 9.75e+11
  - paymentservice-0 (pod) — 102 segments; earliest 01:00; max severity ≈ 6.75e+11
  - node-3 (node) — 163 segments; earliest 01:00; max severity ≈ 3.0e+09
  - checkoutservice-2 (pod) — 349 segments; earliest 01:00; max severity ≈ 1.5e+09
  - (Per-cmdb summary table (top 20) is stored in kernel variable per_cmdb_summary.)

- Timing and onset
  - Many components’ earliest faults are at 2022-03-21 01:00:00+08 (incident window start), suggesting a synchronized onset across many metrics.

- Stored outputs for follow-up
  - Full fault-segment table: metric_fault_segments_full (9,636 rows).
  - Display-ready top rows: metric_fault_segments_head20.
  - Per-component summary: per_cmdb_summary.

If you want, I can (pick one):
- show the top 20 fault segments as a compact table again,
- filter faults to node-level only,
- or correlate these metric faults with trace or log anomalies for the same components/times.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id                                           kpi_name       fault_start_ts_UTC+8         fault_end_ts_UTC+8  duration_minutes  points_in_segment  max_value_in_segment  min_value_in_segment     p95       p5  severity_metric
0   productcatalogservice-2  mesh.destination.checkoutservice.productcatalo...  2022-03-21 01:29:00+08:00  2022-03-21 01:30:00+08:00                 2                  2                6525.0                   0.0  5075.0  1450.00     1.450000e+12
1            emailservice-2  mesh.source.emailservice.jaeger-collector.isti...  2022-03-21 01:10:00+08:00  2022-03-21 01:10:00+08:00                 1                  1                   0.0                   0.0  3900.0   975.00     9.750000e+11
2          paymentservice-0  mesh.destination.checkoutservice.paymentservic...  2022-03-21 01:03:00+08:00  2022-03-21 01:06:00+08:00                 4                  4                3375.0                   0.0  3375.0   675.00     6.750000e+11
3   productcatalogservice-2  mesh.destination.checkoutservice.productcatalo...  2022-03-21 01:29:00+08:00  2022-03-21 01:30:00+08:00                 2                  2                2947.5                   0.0  2292.5   655.00     6.550000e+11
4            emailservice-2  mesh.destination.checkoutservice.emailservice....  2022-03-21 01:10:00+08:00  2022-03-21 01:11:00+08:00                 2                  2                2500.0                   0.0  2500.0   625.00     6.250000e+11
5            emailservice-2  mesh.destination.checkoutservice.emailservice....  2022-03-21 01:10:00+08:00  2022-03-21 01:10:00+08:00                 1                  1                   0.0                   0.0  1730.0   432.50     4.325000e+11
6          paymentservice-0  mesh.destination.checkoutservice.paymentservic...  2022-03-21 01:03:00+08:00  2022-03-21 01:06:00+08:00                 4                  4                1662.5                   0.0  1662.5   332.50     3.325000e+11
7            emailservice-2  mesh.source.emailservice.jaeger-collector.isti...  2022-03-21 01:10:00+08:00  2022-03-21 01:10:00+08:00                 1                  1                   0.0                   0.0   199.0    49.75     4.975000e+10
8                    node-3                                      system.io.w_s  2022-03-21 01:17:00+08:00  2022-03-21 01:19:00+08:00                 3                  3                   3.0                   0.0    28.5     3.00     3.000000e+09
9         checkoutservice-2         container.node-5.container_cpu_cfs_periods  2022-03-21 01:23:00+08:00  2022-03-21 01:23:00+08:00                 1                  1                   0.0                   0.0    10.5     1.50     1.500000e+09
10        checkoutservice-2  mesh.source.checkoutservice.currencyservice.is...  2022-03-21 01:19:00+08:00  2022-03-21 01:19:00+08:00                 1                  1                   0.0                   0.0     6.0     1.50     1.500000e+09
11        checkoutservice-2  mesh.source.checkoutservice.currencyservice.is...  2022-03-21 01:19:00+08:00  2022-03-21 01:19:00+08:00                 1                  1                   0.0                   0.0     6.0     1.50     1.500000e+09
12        checkoutservice-2  mesh.source.checkoutservice.currencyservice.is...  2022-03-21 01:19:00+08:00  2022-03-21 01:19:00+08:00                 1                  1                   0.0                   0.0     6.0     1.50     1.500000e+09
13         paymentservice-1         container.node-6.container_cpu_cfs_periods  2022-03-21 01:16:00+08:00  2022-03-21 01:18:00+08:00                 3                  3                   5.0                   0.0     5.0     1.00     1.000000e+09
14  productcatalogservice-2  mesh.destination.checkoutservice.productcatalo...  2022-03-21 01:29:00+08:00  2022-03-21 01:30:00+08:00                 2                  2                   5.0                   0.0     3.5     1.00     1.000000e+09
15  productcatalogservice-2  mesh.destination.checkoutservice.productcatalo...  2022-03-21 01:29:00+08:00  2022-03-21 01:30:00+08:00                 2                  2                   5.0                   0.0     3.5     1.00     1.000000e+09
16  productcatalogservice-2  mesh.destination.checkoutservice.productcatalo...  2022-03-21 01:29:00+08:00  2022-03-21 01:30:00+08:00                 2                  2                   5.0                   0.0     3.5     1.00     1.000000e+09
17        checkoutservice-2  mesh.source.checkoutservice.cartservice.istio_...  2022-03-21 01:19:00+08:00  2022-03-21 01:19:00+08:00                 1                  1                   0.0                   0.0     4.0     1.00     1.000000e+09
18        checkoutservice-2  mesh.source.checkoutservice.cartservice.istio_...  2022-03-21 01:19:00+08:00  2022-03-21 01:19:00+08:00                 1                  1                   0.0                   0.0     4.0     1.00     1.000000e+09
19        checkoutservice-2  mesh.source.checkoutservice.cartservice.istio_...  2022-03-21 01:19:00+08:00  2022-03-21 01:19:00+08:00                 1                  1                   0.0                   0.0     4.0     1.00     1.000000e+09,                     cmdb_id level  total_fault_segments_in_window  earliest_fault_start_UTC8  max_severity_among_its_faults
0   productcatalogservice-2   pod                             275  2022-03-21 01:00:00+08:00                   1.450000e+12
1            emailservice-2   pod                             154  2022-03-21 01:00:00+08:00                   9.750000e+11
2          paymentservice-0   pod                             102  2022-03-21 01:00:00+08:00                   6.750000e+11
3                    node-3  node                             163  2022-03-21 01:00:00+08:00                   3.000000e+09
4         checkoutservice-2   pod                             349  2022-03-21 01:00:00+08:00                   1.500000e+09
5          paymentservice-1   pod                             109  2022-03-21 01:00:00+08:00                   1.000000e+09
6         checkoutservice-0   pod                             261  2022-03-21 01:00:00+08:00                   5.000000e+06
7         checkoutservice-1   pod                             415  2022-03-21 01:00:00+08:00                   5.000000e+06
8                frontend-1   pod                             384  2022-03-21 01:00:00+08:00                   5.250000e+05
9          paymentservice-2   pod                             127  2022-03-21 01:00:00+08:00                   7.388850e+04
10                   node-1  node                             161  2022-03-21 01:00:00+08:00                   8.988889e+01
11                   node-5  node                             141  2022-03-21 01:00:00+08:00                   7.420000e+01
12       checkoutservice2-0   pod                             421  2022-03-21 01:00:00+08:00                   4.333333e+01
13  productcatalogservice-1   pod                             264  2022-03-21 01:00:00+08:00                   3.300000e+01
14        shippingservice-1   pod                             158  2022-03-21 01:00:00+08:00                   3.300000e+01
15        currencyservice-2   pod                             161  2022-03-21 01:00:00+08:00                   2.539660e+01
16           emailservice-0   pod                             152  2022-03-21 01:00:00+08:00                   2.094848e+01
17           emailservice-1   pod                             137  2022-03-21 01:00:00+08:00                   2.014365e+01
18        shippingservice-0   pod                             150  2022-03-21 01:00:00+08:00                   1.757853e+01
19       shippingservice2-0   pod                             158  2022-03-21 01:00:00+08:00                   1.717914e+01,                       cmdb_id                                           kpi_name       fault_start_ts_UTC+8         fault_end_ts_UTC+8  duration_minutes  points_in_segment  max_value_in_segment  min_value_in_segment          p95      p5  severity_metric
0     productcatalogservice-2  mesh.destination.checkoutservice.productcatalo...  2022-03-21 01:29:00+08:00  2022-03-21 01:30:00+08:00                 2                  2                6525.0                   0.0  5075.000000  1450.0     1.450000e+12
1              emailservice-2  mesh.source.emailservice.jaeger-collector.isti...  2022-03-21 01:10:00+08:00  2022-03-21 01:10:00+08:00                 1                  1                   0.0                   0.0  3900.000000   975.0     9.750000e+11
2            paymentservice-0  mesh.destination.checkoutservice.paymentservic...  2022-03-21 01:03:00+08:00  2022-03-21 01:06:00+08:00                 4                  4                3375.0                   0.0  3375.000000   675.0     6.750000e+11
3     productcatalogservice-2  mesh.destination.checkoutservice.productcatalo...  2022-03-21 01:29:00+08:00  2022-03-21 01:30:00+08:00                 2                  2                2947.5                   0.0  2292.500000   655.0     6.550000e+11
4              emailservice-2  mesh.destination.checkoutservice.emailservice....  2022-03-21 01:10:00+08:00  2022-03-21 01:11:00+08:00                 2                  2                2500.0                   0.0  2500.000000   625.0     6.250000e+11
...                       ...                                                ...                        ...                        ...               ...                ...                   ...                   ...          ...     ...              ...
9631       shippingservice2-0        container.node-6.container_cpu_user_seconds  2022-03-21 01:28:00+08:00  2022-03-21 01:28:00+08:00                 1                  1                   0.0                   0.0     0.015000     0.0     0.000000e+00
9632       shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  2022-03-21 01:16:00+08:00  2022-03-21 01:16:00+08:00                 1                  1                   0.0                   0.0     0.001575     0.0     0.000000e+00
9633       shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  2022-03-21 01:26:00+08:00  2022-03-21 01:26:00+08:00                 1                  1                   0.0                   0.0     0.001575     0.0     0.000000e+00
9634       shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  2022-03-21 01:28:00+08:00  2022-03-21 01:28:00+08:00                 1                  1                   0.0                   0.0     0.001575     0.0     0.000000e+00
9635       shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...  2022-03-21 01:30:00+08:00  2022-03-21 01:30:00+08:00                 1                  1                   0.0                   0.0     0.001575     0.0     0.000000e+00

[9636 rows x 11 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Reuse existing kernel variables
try:
    metric_fault_segments_full
except NameError:
    metric_fault_segments_full = metric_fault_segments_full = pd.DataFrame()  # fallback empty

try:
    trace_incidents_agg
except NameError:
    trace_incidents_agg = pd.DataFrame()

tz = pytz.timezone('Asia/Shanghai')

# Helper to ensure datetime tz-aware minute column
def ensure_minute_column(df, ts_col, minute_col='minute_UTC+8'):
    if ts_col not in df.columns:
        df[minute_col] = pd.NaT
        return df
    # parse to datetime, preserve timezone if present, else assume UTC+8
    dt = pd.to_datetime(df[ts_col], errors='coerce')
    # If timezone-naive, localize to Asia/Shanghai; if tz-aware, convert to Asia/Shanghai
    if dt.dt.tz is None:
        dt = dt.dt.tz_localize(tz)
    else:
        dt = dt.dt.tz_convert(tz)
    # floor to minute
    df[minute_col] = dt.dt.floor('T')
    return df

# 1) Per-minute histogram of metric fault segments by fault_start_ts_UTC+8
mfs = metric_fault_segments_full.copy()
mfs = ensure_minute_column(mfs, 'fault_start_ts_UTC+8', 'minute_UTC+8')
metric_minute_hist = (
    mfs.groupby('minute_UTC+8')
       .size()
       .reset_index(name='fault_segments_count')
       .sort_values('fault_segments_count', ascending=False)
       .head(10)
       .reset_index(drop=True)
)

# 2) For each of those top 10 minutes, top 5 cmdb_id by segments starting at that minute
top_minutes = metric_minute_hist['minute_UTC+8'].dropna().tolist()
rows = []
for minute in top_minutes:
    # filter segments starting at this minute
    segs_at_min = mfs[mfs['minute_UTC+8'] == minute]
    top_cmdb = (
        segs_at_min.groupby('cmdb_id')
                   .size()
                   .reset_index(name='segments_starting_at_minute')
                   .sort_values('segments_starting_at_minute', ascending=False)
                   .head(5)
    )
    if not top_cmdb.empty:
        top_cmdb['minute_UTC+8'] = minute
        # reorder columns
        top_cmdb = top_cmdb[['minute_UTC+8','cmdb_id','segments_starting_at_minute']]
        rows.append(top_cmdb)
# concatenate results (may be empty)
metric_top_components_per_min = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['minute_UTC+8','cmdb_id','segments_starting_at_minute'])

# 3) Per-minute histogram of trace anomalies by earliest_anomaly_ts_UTC+8
tr = trace_incidents_agg.copy()
tr = ensure_minute_column(tr, 'earliest_anomaly_ts_UTC+8', 'minute_UTC+8')
trace_minute_hist = (
    tr.groupby('minute_UTC+8')
      .size()
      .reset_index(name='trace_anomaly_count')
      .sort_values('trace_anomaly_count', ascending=False)
      .head(10)
      .reset_index(drop=True)
)

# 4) For top 2 metric-minute peaks, return distinct fault segments starting at those minutes sorted by severity desc (top 20)
top2_minutes = metric_minute_hist['minute_UTC+8'].dropna().head(2).tolist()
segments_list = []
for minute in top2_minutes:
    segs = mfs[mfs['minute_UTC+8'] == minute].copy()
    if segs.empty:
        continue
    # ensure numeric severity_metric
    segs['severity_metric'] = pd.to_numeric(segs.get('severity_metric', 0), errors='coerce').fillna(0)
    sel = segs[[
        'cmdb_id','kpi_name','fault_start_ts_UTC+8','duration_minutes','points_in_segment',
        'max_value_in_segment','p95','severity_metric'
    ]].sort_values(['severity_metric','duration_minutes'], ascending=[False,False]).head(20)
    # annotate the minute for clarity
    sel['peak_minute_UTC+8'] = minute
    segments_list.append(sel)
metric_top_segments_peaks = pd.concat(segments_list, ignore_index=True) if segments_list else pd.DataFrame(columns=[
    'cmdb_id','kpi_name','fault_start_ts_UTC+8','duration_minutes','points_in_segment',
    'max_value_in_segment','p95','severity_metric','peak_minute_UTC+8'
])

# Keep outputs compact for display: show the computed DataFrames (already limited)
metric_minute_hist, metric_top_components_per_min, trace_minute_hist, metric_top_segments_peaks

```
Out[10]:
```
Summary of results (plain English):

1) Metric fault-segment minute histogram (top 10 minutes by segments starting that minute)
- 2022-03-21 01:00:00+08 — 3,649 segments
- 2022-03-21 01:14:00+08 — 389 segments
- 2022-03-21 01:27:00+08 — 304 segments
- 2022-03-21 01:26:00+08 — 290 segments
- 2022-03-21 01:30:00+08 — 289 segments
- 2022-03-21 01:09:00+08 — 265 segments
- 2022-03-21 01:28:00+08 — 260 segments
- 2022-03-21 01:21:00+08 — 253 segments
- 2022-03-21 01:20:00+08 — 242 segments
- 2022-03-21 01:16:00+08 — 240 segments

2) Top components starting faults at those minutes (top 5 per minute; examples)
- For the primary peak (2022-03-21 01:00:00+08) the top 5 components starting segments are:
  - adservice — 316 segments
  - frontend2-0 — 139 segments
  - frontend-2 — 136 segments
  - checkoutservice-0 — 132 segments
  - frontend-1 — 131 segments
- For other minutes there are similar ranked lists (example for 01:16 in the computed table: checkoutservice-2 (50), checkoutservice-1 (27), productcatalogservice-2 (20), frontend-2 (15), productcatalogservice-0 (11)).
- Full top-5-per-minute results for the top 10 minutes are stored in metric_top_components_per_min.

3) Trace-anomaly minute histogram (top 10)
- Top minute for trace anomalies is 2022-03-21 01:00:00+08 with 20 trace groups flagged (this was the dominant minute for traces as well).

4) Fault segments for the two metric peaks (01:00 and 01:14)
- For 2022-03-21 01:00:00+08 (primary peak) the highest-severity segments starting at that minute include:
  - node-5 / system.io.avg_q_sz — start 01:00, duration 8 min, points 8, max=3.71, p95=0.05, severity ≈ 74.2
  - checkoutservice-2 / container.node-5.container_memory_failures.* — start 01:00, duration 31 min (31 points), max=33.0, p95=0.0, severity ≈ 33.0
  - frontend-1 / container.node-5.container_memory_failures.* — similar high-severity segments starting 01:00
  (Many other high-severity segments start at 01:00; the table metric_top_segments_peaks contains the top 20 by severity for the top 2 minutes.)
- For 2022-03-21 01:14:00+08 (secondary peak) notable high-severity trace/mesh/CPU metrics appear, e.g. shippingservice-1 and currencyservice-* entries with severity ≈1.11–1.13 for certain mesh/cpu metrics.

5) Where to find the detailed tables (stored for follow-up)
- metric_minute_hist — per-minute counts of metric fault segments (top 10 shown above)
- metric_top_components_per_min — top-5 components per top minute (full compact table stored)
- trace_minute_hist — per-minute counts of trace anomalies (top minute shown above)
- metric_top_segments_peaks — top fault segments (up to 20) starting at the two highest metric-minute peaks, sorted by severity

Takeaway:
- A strong, system-wide spike occurs at 2022-03-21 01:00+08 (largest single-minute hotspot with 3,649 metric fault segments and the most trace anomalies), with many components (adservice, multiple frontends, checkoutservice pods, nodes) affected at that exact minute. This minute is the primary focal point for further correlation (logs/traces) and root-cause investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(               minute_UTC+8  fault_segments_count
0 2022-03-21 01:00:00+08:00                  3649
1 2022-03-21 01:14:00+08:00                   389
2 2022-03-21 01:27:00+08:00                   304
3 2022-03-21 01:26:00+08:00                   290
4 2022-03-21 01:30:00+08:00                   289
5 2022-03-21 01:09:00+08:00                   265
6 2022-03-21 01:28:00+08:00                   260
7 2022-03-21 01:21:00+08:00                   253
8 2022-03-21 01:20:00+08:00                   242
9 2022-03-21 01:16:00+08:00                   240,                 minute_UTC+8                  cmdb_id  segments_starting_at_minute
0  2022-03-21 01:00:00+08:00                adservice                          316
1  2022-03-21 01:00:00+08:00              frontend2-0                          139
2  2022-03-21 01:00:00+08:00               frontend-2                          136
3  2022-03-21 01:00:00+08:00        checkoutservice-0                          132
4  2022-03-21 01:00:00+08:00               frontend-1                          131
..                       ...                      ...                          ...
45 2022-03-21 01:16:00+08:00        checkoutservice-2                           50
46 2022-03-21 01:16:00+08:00        checkoutservice-1                           27
47 2022-03-21 01:16:00+08:00  productcatalogservice-2                           20
48 2022-03-21 01:16:00+08:00               frontend-2                           15
49 2022-03-21 01:16:00+08:00  productcatalogservice-0                           11

[50 rows x 3 columns],                minute_UTC+8  trace_anomaly_count
0 2022-03-21 01:00:00+08:00                   20,               cmdb_id                                           kpi_name       fault_start_ts_UTC+8  duration_minutes  points_in_segment  max_value_in_segment           p95  severity_metric         peak_minute_UTC+8
0              node-5                                 system.io.avg_q_sz  2022-03-21 01:00:00+08:00                 8                  8              3.710000      0.050000        74.200000 2022-03-21 01:00:00+08:00
1   checkoutservice-2  container.node-5.container_memory_failures.con...  2022-03-21 01:00:00+08:00                31                 31             33.000000      0.000000        33.000000 2022-03-21 01:00:00+08:00
2   checkoutservice-2  container.node-5.container_memory_failures.hie...  2022-03-21 01:00:00+08:00                31                 31             33.000000      0.000000        33.000000 2022-03-21 01:00:00+08:00
3          frontend-1  container.node-5.container_memory_failures.con...  2022-03-21 01:00:00+08:00                31                 31             33.000000      0.000000        33.000000 2022-03-21 01:00:00+08:00
4          frontend-1  container.node-5.container_memory_failures.hie...  2022-03-21 01:00:00+08:00                31                 31             33.000000      0.000000        33.000000 2022-03-21 01:00:00+08:00
..                ...                                                ...                        ...               ...                ...                   ...           ...              ...                       ...
35  shippingservice-1  mesh.source.shippingservice.jaeger-collector.i...  2022-03-21 01:14:00+08:00                 2                  2          27850.000000  27825.000000         1.129562 2022-03-21 01:14:00+08:00
36  currencyservice-1       container.node-6.container_cpu_usage_seconds  2022-03-21 01:14:00+08:00                 1                  1              0.135758      0.121062         1.121390 2022-03-21 01:14:00+08:00
37  currencyservice-1         container.node-6.container_cpu_cfs_periods  2022-03-21 01:14:00+08:00                 1                  1            148.500000    132.500000         1.120755 2022-03-21 01:14:00+08:00
38  shippingservice-1         container.node-6.container_cpu_cfs_periods  2022-03-21 01:14:00+08:00                 1                  1             24.500000     22.000000         1.113636 2022-03-21 01:14:00+08:00
39         frontend-0  mesh.source.frontend.shippingservice.istio_req...  2022-03-21 01:14:00+08:00                 2                  2          17050.000000  16275.000000         1.111111 2022-03-21 01:14:00+08:00

[40 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables if present
try:
    metric_fault_segments_full
except NameError:
    metric_fault_segments_full = pd.DataFrame()

try:
    trace_incidents_agg
except NameError:
    trace_incidents_agg = pd.DataFrame()

# Timezone
tz = pytz.timezone('Asia/Shanghai')

# Helper to ensure minute column exists and is tz-aware Asia/Shanghai
def ensure_minute_column(df, ts_col, minute_col='minute_UTC+8'):
    df = df.copy()
    if ts_col not in df.columns or df.shape[0] == 0:
        df[minute_col] = pd.NaT
        return df
    dt = pd.to_datetime(df[ts_col], errors='coerce')
    # If dt is timezone-naive, localize to Asia/Shanghai; if tz-aware, convert to Asia/Shanghai
    if getattr(dt.dt, 'tz', None) is None:
        # some pandas versions return None for tz attribute; check first element for tzinfo
        sample = dt.iloc[0]
        if hasattr(sample, 'tzinfo') and sample.tzinfo is not None:
            dt = dt.dt.tz_convert(tz)
        else:
            dt = dt.dt.tz_localize(tz)
    else:
        # convert to target tz
        dt = dt.dt.tz_convert(tz)
    df[minute_col] = dt.dt.floor('T')
    return df

# Prepare metric and trace frames with minute columns
mfs = ensure_minute_column(metric_fault_segments_full, 'fault_start_ts_UTC+8', 'minute_UTC+8')
# Ensure numeric severity_metric
if 'severity_metric' in mfs.columns:
    mfs['severity_metric'] = pd.to_numeric(mfs['severity_metric'], errors='coerce').fillna(0.0)
else:
    mfs['severity_metric'] = 0.0

tr = ensure_minute_column(trace_incidents_agg, 'earliest_anomaly_ts_UTC+8', 'minute_UTC+8')
# Ensure trace severity numeric
if 'severity' in tr.columns:
    tr['severity'] = pd.to_numeric(tr['severity'], errors='coerce').fillna(0.0)
else:
    tr['severity'] = 0.0

# Define minutes of interest
minute_1 = tz.localize(pd.to_datetime("2022-03-21 01:00:00"))
minute_2 = tz.localize(pd.to_datetime("2022-03-21 01:14:00"))
minutes = [minute_1, minute_2]

# Service and node sets for level inference
service_names = set([
    'frontend','shippingservice','checkoutservice','currencyservice','adservice','emailservice',
    'cartservice','productcatalogservice','recommendationservice','paymentservice'
])
nodes_set = set([f'node-{i}' for i in range(1,7)])

def determine_level(name):
    if name in nodes_set:
        return 'node'
    if name in service_names:
        return 'service'
    if isinstance(name, str) and '-' in name:
        return 'pod'
    return 'service'

def build_fusion_for_minute(minute_dt):
    # filter metric segments starting exactly at this minute
    m_at = mfs[mfs['minute_UTC+8'] == minute_dt].copy()
    # count and max severity per cmdb_id
    m_summary = m_at.groupby('cmdb_id').agg(
        metric_faults_count_at_minute = ('kpi_name','size'),
        max_metric_severity_at_minute = ('severity_metric','max')
    ).reset_index()
    # filter trace rows with earliest anomaly at this minute AND anomaly_count>0
    t_at = tr[(tr['minute_UTC+8'] == minute_dt) & (tr.get('anomaly_count', tr.get('anomaly_count', pd.Series([0]*len(tr)))) > 0)].copy()
    # if anomaly_count column missing, fall back to counting all rows at minute
    if 'anomaly_count' not in t_at.columns:
        t_at = tr[tr['minute_UTC+8'] == minute_dt].copy()
    t_summary = t_at.groupby('cmdb_id').agg(
        trace_groups_count_at_minute = ('trace_name','size'),
        max_trace_severity_at_minute = ('severity','max')
    ).reset_index()
    # union of cmdb_ids present in either
    cmdbs = pd.Series(list(set(m_summary['cmdb_id'].tolist() + t_summary['cmdb_id'].tolist())))
    if cmdbs.empty:
        return pd.DataFrame(columns=[
            'minute_UTC+8','cmdb_id','level','metric_faults_count_at_minute','max_metric_severity_at_minute',
            'trace_groups_count_at_minute','max_trace_severity_at_minute','earliest_fault_or_trace_ts'
        ])
    df = pd.DataFrame({'cmdb_id': cmdbs})
    df = df.merge(m_summary, on='cmdb_id', how='left')
    df = df.merge(t_summary, on='cmdb_id', how='left')
    df['metric_faults_count_at_minute'] = df['metric_faults_count_at_minute'].fillna(0).astype(int)
    df['max_metric_severity_at_minute'] = pd.to_numeric(df['max_metric_severity_at_minute'], errors='coerce').fillna(0.0)
    df['trace_groups_count_at_minute'] = df['trace_groups_count_at_minute'].fillna(0).astype(int)
    df['max_trace_severity_at_minute'] = pd.to_numeric(df['max_trace_severity_at_minute'], errors='coerce').fillna(0.0)
    df['level'] = df['cmdb_id'].apply(determine_level)
    df['minute_UTC+8'] = minute_dt
    df['earliest_fault_or_trace_ts'] = minute_dt  # per spec, the minute itself
    # sort by combined max severity then by metric faults count
    df['combined_max_sev'] = df[['max_metric_severity_at_minute','max_trace_severity_at_minute']].max(axis=1)
    df = df.sort_values(['combined_max_sev','metric_faults_count_at_minute'], ascending=[False,False]).reset_index(drop=True)
    df = df.drop(columns=['combined_max_sev'])
    return df.head(20)

# Build fusion tables for the two minutes and store in kernel variables
fusion_peak_01_00 = build_fusion_for_minute(minute_1)
fusion_peak_01_14 = build_fusion_for_minute(minute_2)

# Keep outputs compact (top 20 rows per minute)
fusion_peak_01_00, fusion_peak_01_14

```
Out[11]:
```
Summary (plain English):

I fused metric fault segments and trace anomalies for the two peak minutes 2022-03-21 01:00:00+08 and 2022-03-21 01:14:00+08. Results (top rows shown):

1) Minute = 2022-03-21 01:00:00+08 (primary peak)
- Many components showed metric faults starting at this minute; highest-ranked by combined severity:
  - node-5 (level=node): 33 metric faults, max_metric_severity ≈ 74.20, no trace groups
  - frontend-1 (pod): 131 metric faults, max_metric_severity = 33.00
  - productcatalogservice-1 (pod): 90 metric faults, max_metric_severity = 33.00
  - checkoutservice-2 (pod): 84 metric faults, max_metric_severity = 33.00
  - shippingservice-1 (pod): 55 metric faults, max_metric_severity = 33.00, trace_groups_count = 1 (max_trace_severity = 2.0)
  - currencyservice-2 (pod): 71 metric faults, max_metric_severity ≈ 25.40
  - emailservice-2 (pod): 64 metric faults, max_metric_severity ≈ 21.62
  - checkoutservice-0 (pod): 132 metric faults, max_metric_severity = 16.50
  - frontend-0, frontend-2 and other pods also appear in the top 20 for this minute.
- earliest_fault_or_trace_ts for all rows is the minute itself (01:00).

Interpretation for 01:00:
- Node- and pod-level metric anomalies are widespread at the exact same minute. Node-5 stands out with the largest metric severity, and many productcatalog/checkout/frontend pods also exhibit high-severity metric faults. A small number of trace groups align to the minute (e.g., shippingservice-1).

2) Minute = 2022-03-21 01:14:00+08 (secondary peak)
- Top components at this minute (sorted by severity then count) include:
  - node-3 (node): 6 metric faults, max_metric_severity ≈ 2.88
  - shippingservice-2 (pod): 5 metric faults, max_metric_severity ≈ 1.95
  - cartservice-0 (pod): 17 metric faults, max_metric_severity ≈ 1.93
  - emailservice-1 (pod): 1 metric fault, max_metric_severity = 1.50
  - frontend-1 (pod): 29 metric faults, max_metric_severity ≈ 1.44
  - Other pods/services (shippingservice-1, checkoutservice, paymentservice2-0, etc.) are also present.
- trace_groups_count at this minute is zero for most top entries (trace anomalies were concentrated strongly at 01:00).

Interpretation for 01:14:
- A smaller secondary hotspot exists with moderate severities. It involves different nodes/pods than the primary minute (node-3, shippingservice-2, cartservice-0, etc.).

Stored outputs for follow-up:
- fusion_peak_01_00 and fusion_peak_01_14 were saved in the kernel (each contains up to the top 20 rows per minute). Use these for targeted log/trace correlation and deeper root-cause analysis.

Bottom line / next steps:
- Focus correlation efforts first on the primary minute 2022-03-21 01:00+08. Node-5 and many productcatalog/checkout/frontend pods show the strongest metric severities at that minute and are top candidates for root-cause investigation. The secondary peak at 01:14+08 is relevant but less severe and affects a slightly different set of components.

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  metric_faults_count_at_minute  max_metric_severity_at_minute  trace_groups_count_at_minute  max_trace_severity_at_minute level              minute_UTC+8 earliest_fault_or_trace_ts
0                     node-5                             33                      74.200000                             0                      0.000000  node 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
1                 frontend-1                            131                      33.000000                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
2    productcatalogservice-1                             90                      33.000000                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
3          checkoutservice-2                             84                      33.000000                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
4          shippingservice-1                             55                      33.000000                             1                      2.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
5          currencyservice-2                             71                      25.396598                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
6             emailservice-2                             64                      21.618047                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
7          checkoutservice-0                            132                      16.500000                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
8   productcatalogservice2-0                             72                       9.000000                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
9    productcatalogservice-2                             77                       6.881080                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
10                    node-6                             39                       6.500000                             0                      0.000000  node 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
11        currencyservice2-0                             56                       2.500000                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
12   productcatalogservice-0                             91                       2.209815                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
13         shippingservice-0                             56                       2.000000                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
14                frontend-0                            126                       1.037422                             1                      2.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
15          paymentservice-1                             56                       2.000000                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
16             cartservice-2                             69                       1.000000                             1                      1.873278   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
17        checkoutservice2-0                            113                       1.843053                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
18   recommendationservice-0                             77                       1.709164                             0                      0.000000   pod 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00
19                    node-4                             34                       1.500000                             0                      0.000000  node 2022-03-21 01:00:00+08:00  2022-03-21 01:00:00+08:00,                     cmdb_id  metric_faults_count_at_minute  max_metric_severity_at_minute  trace_groups_count_at_minute  max_trace_severity_at_minute    level              minute_UTC+8 earliest_fault_or_trace_ts
0                    node-3                              6                       2.884393                             0                           0.0     node 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
1         shippingservice-2                              5                       1.952381                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
2             cartservice-0                             17                       1.931604                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
3            emailservice-1                              1                       1.500000                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
4                frontend-1                             29                       1.441467                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
5         shippingservice-1                              9                       1.329082                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
6           checkoutservice                              1                       1.272527                             0                           0.0  service 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
7         shippingservice-0                              5                       1.200000                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
8              emailservice                              2                       1.200000                             0                           0.0  service 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
9         paymentservice2-0                              4                       1.199331                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
10              cartservice                              2                       1.165625                             0                           0.0  service 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
11        currencyservice-1                              5                       1.121390                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
12               frontend-0                              8                       1.111111                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
13        checkoutservice-1                             45                       1.104885                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
14  recommendationservice-1                              1                       1.098004                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
15       checkoutservice2-0                             12                       1.097416                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
16        currencyservice-0                              7                       1.095238                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
17        checkoutservice-2                             45                       1.091437                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
18  productcatalogservice-1                             24                       1.088589                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00
19            cartservice-2                              5                       1.079787                             0                           0.0      pod 2022-03-21 01:14:00+08:00  2022-03-21 01:14:00+08:00)```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

# Reuse kernel variables (assumed present)
try:
    mfs_full = metric_fault_segments_full
except NameError:
    mfs_full = pd.DataFrame()

try:
    tr_inc = trace_incidents_agg
except NameError:
    tr_inc = pd.DataFrame()

tz = pytz.timezone('Asia/Shanghai')

# Helper to parse timestamp strings to timezone-aware Asia/Shanghai datetimes
def to_sh(series):
    dt = pd.to_datetime(series, errors='coerce')
    # If tz-naive, localize; if tz-aware, convert
    if not dt.empty:
        if dt.dt.tz is None:
            dt = dt.dt.tz_localize(tz)
        else:
            dt = dt.dt.tz_convert(tz)
    return dt

# Define the two peak minutes as tz-aware datetimes
minute_01_00 = tz.localize(pd.to_datetime("2022-03-21 01:00:00"))
minute_01_14 = tz.localize(pd.to_datetime("2022-03-21 01:14:00"))

# Ensure mfs_full has parsed fault_start datetime column for robust comparison
if 'fault_start_ts_UTC+8_parsed' not in mfs_full.columns:
    if 'fault_start_ts_UTC+8' in mfs_full.columns:
        mfs_full = mfs_full.copy()
        mfs_full['fault_start_ts_UTC+8_parsed'] = to_sh(mfs_full['fault_start_ts_UTC+8'])
    else:
        mfs_full = mfs_full.copy()
        mfs_full['fault_start_ts_UTC+8_parsed'] = pd.NaT

# 1) Node-5 metric fault segments at 2022-03-21 01:00:00+08:00 (up to 20 rows)
node5_metrics = pd.DataFrame()
if not mfs_full.empty:
    node5_metrics = mfs_full[
        (mfs_full['cmdb_id'] == 'node-5') &
        (mfs_full['fault_start_ts_UTC+8_parsed'] == minute_01_00)
    ]
    # select and keep columns compact
    cols = ['kpi_name','fault_start_ts_UTC+8','duration_minutes','points_in_segment','max_value_in_segment','p95','severity_metric']
    node5_metrics = node5_metrics.loc[:, [c for c in cols if c in node5_metrics.columns]].head(20)

# 2) Node-5: pods that had metric fault segments at that minute referencing '.node-5.' in kpi_name
node5_pods = pd.DataFrame()
if not mfs_full.empty:
    pods_mask = (
        (mfs_full['fault_start_ts_UTC+8_parsed'] == minute_01_00) &
        (mfs_full['kpi_name'].astype(str).str.contains(r'\.node-5\.', regex=False))
    )
    pods_df = mfs_full.loc[pods_mask, ['cmdb_id']].copy()
    if not pods_df.empty:
        # exclude the node itself if present
        pods_df = pods_df[pods_df['cmdb_id'] != 'node-5']
        node5_pods = pods_df.groupby('cmdb_id').size().reset_index(name='segments_count').sort_values('segments_count', ascending=False).head(20).reset_index(drop=True)

# 3) Node-3 metric fault segments at 2022-03-21 01:14:00+08:00 (up to 20 rows)
node3_metrics = pd.DataFrame()
if not mfs_full.empty:
    node3_metrics = mfs_full[
        (mfs_full['cmdb_id'] == 'node-3') &
        (mfs_full['fault_start_ts_UTC+8_parsed'] == minute_01_14)
    ]
    cols = ['kpi_name','fault_start_ts_UTC+8','duration_minutes','points_in_segment','max_value_in_segment','p95','severity_metric']
    node3_metrics = node3_metrics.loc[:, [c for c in cols if c in node3_metrics.columns]].head(20)

# 4) Node-3: pods that had metric fault segments at that minute referencing '.node-3.' in kpi_name
node3_pods = pd.DataFrame()
if not mfs_full.empty:
    pods_mask3 = (
        (mfs_full['fault_start_ts_UTC+8_parsed'] == minute_01_14) &
        (mfs_full['kpi_name'].astype(str).str.contains(r'\.node-3\.', regex=False))
    )
    pods_df3 = mfs_full.loc[pods_mask3, ['cmdb_id']].copy()
    if not pods_df3.empty:
        pods_df3 = pods_df3[pods_df3['cmdb_id'] != 'node-3']
        node3_pods = pods_df3.groupby('cmdb_id').size().reset_index(name='segments_count').sort_values('segments_count', ascending=False).head(20).reset_index(drop=True)

# 5) Trace rows for both minutes (combined, up to 50 rows)
trace_peak_rows = pd.DataFrame()
if not tr_inc.empty:
    # ensure parsed earliest timestamp column
    if 'earliest_parsed' not in tr_inc.columns:
        tr_inc = tr_inc.copy()
        if 'earliest_anomaly_ts_UTC+8' in tr_inc.columns:
            tr_inc['earliest_parsed'] = to_sh(tr_inc['earliest_anomaly_ts_UTC+8'])
        else:
            tr_inc['earliest_parsed'] = pd.NaT
    trace_mask = tr_inc['earliest_parsed'].isin([minute_01_00, minute_01_14])
    trace_peak_rows = tr_inc.loc[trace_mask, [
        c for c in ['cmdb_id','trace_name','total_points_in_window','anomaly_count','earliest_anomaly_ts_UTC+8','max_value_in_window','p95','severity'] if c in tr_inc.columns
    ]].sort_values(['earliest_anomaly_ts_UTC+8','severity'], ascending=[True,False]).head(50).reset_index(drop=True)

# Store results in kernel variables for follow-up
node5_metrics = node5_metrics
node5_pods = node5_pods
node3_metrics = node3_metrics
node3_pods = node3_pods
trace_peak_rows = trace_peak_rows

# Display compact outputs (variable names only, as IPython will show them)
node5_metrics, node5_pods, node3_metrics, node3_pods, trace_peak_rows

```
Out[12]:
```
Evidence summary (compact, plain English):

1) node-5 at 2022-03-21 01:00:00+08 — metric fault segments (up to 20 shown)
- High-severity I/O metrics start at 01:00 and persist:
  - system.io.avg_q_sz — duration 8 min, points=8, max=3.71, p95=0.05, severity=74.20
  - system.io.rkb_s — duration 31 min, points=31, max=58.0, p95=2.0, severity=29.00
  - system.io.w_await — duration 8 min, max=15.13, p95=0.73, severity≈20.73
  - system.io.await — duration 5 min, max=14.84, p95=0.73, severity≈20.33
  - system.io.util, system.io.r_s, system.cpu.iowait, system.io.w_s, and other disk/io KPIs also flagged (many with 31 points covering the window).
- Interpretation: strong, sustained node-level disk I/O pressure on node-5 beginning at 01:00 — the largest metric severities are I/O-related.

2) node-5 → pods (pods referencing '.node-5.' in kpi_name) at 01:00
- Result: no pod cmdb_id rows were returned (empty). In other words, among the metric fault segments that start exactly at 01:00 with kpi_name containing ".node-5.", the result set is empty (no pod names found in that specific matching). Practical note: pods running on node-5 may still be affected, but the metric kpi_name filters did not yield explicit pod entries referencing ".node-5." for segments starting at that minute in the fault-segment table.

3) node-3 at 2022-03-21 01:14:00+08 — metric fault segments (up to 20 shown)
- Notable CPU/host metrics at 01:14:
  - system.cpu.user — duration 1 min, points=1, max=4.99, p95=1.73, severity≈2.88
  - system.cpu.pct_usage — duration 1 min, max=6.49, p95=3.04, severity≈2.13
  - system.cpu.system, system.load.15, system.fs.inodes.used, system.net.tcp.retrans_segs — present with milder severities (~1.0).
- Interpretation: node-3 shows short-lived CPU/load-related anomalies at 01:14 (moderate severity compared with node-5 I/O at 01:00).

4) node-3 → pods referencing '.node-3.' at 01:14
- Result: no pod cmdb_id rows were returned (empty). As with node-5, no direct pod entries in the fault-segment table contained ".node-3." at that minute.

5) Trace anomalies at the two minutes (combined, top rows)
- Most trace anomalies are concentrated at 01:00:
  - frontend-0 / trace.to_shippingservice-1.row_count — total=30, anomaly_count=30, earliest=01:00, max=3, p95=3, severity=2.0
  - shippingservice-1 / trace.from_frontend-0.row_count — total=30, anomaly_count=30, earliest=01:00, max=3, p95=3, severity=2.0
  - cartservice-2 / trace.self.duration_mean — total=30, anomaly_count=30, earliest=01:00, severity≈1.87
  - Many adservice and cartservice trace.error_rate groups flagged (anomaly_count=30) but with zero-valued series (p95=0), hence zero computed severity.
- Interpretation: application-level traces show synchronized anomalies at 01:00 (row_count and duration metrics), matching the metric spike at the same minute.

Bottom-line supporting evidence for root-cause decision:
- The strongest signal is node-5 at 01:00: extensive, high-severity disk I/O metrics (avg_q_sz, rkb_s, w_await, await, io.util) with long durations (many groups covering 31 points). This strongly indicates node-5 I/O degradation starting at the incident window start.
- Trace anomalies at 01:00 (frontends, shippingservice, cartservice, etc.) show the application-level impact coincident with node-5’s I/O problems — consistent with a node-level I/O root cause that propagated to services.
- node-3 at 01:14 shows CPU/load anomalies but with lower severity and shorter duration — likely a secondary/localized issue.

Stored kernel variables for follow-up:
- node5_metrics (detailed node-5 metric segments at 01:00)
- node5_pods (pod counts referencing '.node-5.' — empty)
- node3_metrics (node-3 metric segments at 01:14)
- node3_pods (pod counts referencing '.node-3.' — empty)
- trace_peak_rows (trace anomalies at the two peak minutes)

Recommendation:
- Prioritize investigation on node-5 I/O (check disk health, host-level logs, kubelet/CRI, and map pods scheduled on node-5 during 01:00 to inspect their application logs).
- Use the stored tables to correlate pod-to-node mapping and container logs for the top affected pods/services (adservice, frontend*, checkoutservice*, productcatalogservice*), and then assess whether mitigation (cordon/drain or disk remediation) is needed.

The original code execution output of IPython Kernel is also provided below for reference:

(                         kpi_name       fault_start_ts_UTC+8  duration_minutes  points_in_segment  max_value_in_segment           p95  severity_metric
54             system.io.avg_q_sz  2022-03-21 01:00:00+08:00                 8                  8          3.710000e+00  5.000000e-02        74.200000
65                system.io.rkb_s  2022-03-21 01:00:00+08:00                31                 31          5.800000e+01  2.000000e+00        29.000000
71              system.io.w_await  2022-03-21 01:00:00+08:00                 8                  8          1.513000e+01  7.300000e-01        20.726027
72                system.io.await  2022-03-21 01:00:00+08:00                 5                  5          1.484000e+01  7.300000e-01        20.328767
95                 system.io.util  2022-03-21 01:00:00+08:00                 5                  5          8.500000e+00  9.500000e-01         8.947368
112                 system.io.r_s  2022-03-21 01:00:00+08:00                31                 31          2.500000e+00  5.000000e-01         5.000000
199             system.cpu.iowait  2022-03-21 01:00:00+08:00                31                 31          4.000000e-02  2.000000e-02         2.000000
285                 system.io.w_s  2022-03-21 01:00:00+08:00                 8                  8          1.250000e+02  6.650000e+01         1.879699
414               system.cpu.user  2022-03-21 01:00:00+08:00                 3                  3          6.020000e+00  4.170000e+00         1.443645
453               system.io.svctm  2022-03-21 01:00:00+08:00                 5                  5          3.400000e-01  2.500000e-01         1.360000
1115       system.udp.connect.num  2022-03-21 01:00:00+08:00                31                 31          1.100000e+01  1.000000e+01         1.100000
1481              system.mem.free  2022-03-21 01:00:00+08:00                 4                  4          1.028000e+03  9.690000e+02         1.060888
3606            system.disk.total  2022-03-21 01:00:00+08:00                31                 31          1.854945e+10  1.854945e+10         1.000000
3607      system.fs.inodes.in_use  2022-03-21 01:00:00+08:00                31                 31          4.700000e-01  4.700000e-01         1.000000
3608             system.mem.total  2022-03-21 01:00:00+08:00                31                 31          3.211700e+04  3.211700e+04         1.000000
3609  system.net.tcp.retrans_segs  2022-03-21 01:00:00+08:00                31                 31          2.000000e-02  2.000000e-02         1.000000
3610         system.os.nofile.max  2022-03-21 01:00:00+08:00                31                 31          6.553600e+06  6.553600e+06         1.000000
3611    system.process.zombie.num  2022-03-21 01:00:00+08:00                31                 31          2.400000e+01  2.400000e+01         1.000000
3612       system.tcp.retrans_pct  2022-03-21 01:00:00+08:00                31                 31          1.000000e-02  1.000000e-02         1.000000
6290         system.cpu.pct_usage  2022-03-21 01:00:00+08:00                 1                  1          6.520000e+00  6.520000e+00         1.000000, Empty DataFrame
Columns: []
Index: [],                          kpi_name       fault_start_ts_UTC+8  duration_minutes  points_in_segment  max_value_in_segment         p95  severity_metric
158               system.cpu.user  2022-03-21 01:14:00+08:00                 1                  1                  4.99        1.73         2.884393
178          system.cpu.pct_usage  2022-03-21 01:14:00+08:00                 1                  1                  6.49        3.04         2.134868
2119            system.cpu.system  2022-03-21 01:14:00+08:00                 1                  1                  1.10        1.08         1.018519
2140               system.load.15  2022-03-21 01:14:00+08:00                 2                  2                  0.57        0.79         1.017857
2877        system.fs.inodes.used  2022-03-21 01:14:00+08:00                 1                  1            3447777.50  3447770.00         1.000002
6239  system.net.tcp.retrans_segs  2022-03-21 01:14:00+08:00                 1                  1                  0.12        0.12         1.000000, Empty DataFrame
Columns: []
Index: [],               cmdb_id                            trace_name  total_points_in_window  anomaly_count  earliest_anomaly_ts_UTC+8  max_value_in_window       p95  severity
0          frontend-0  trace.to_shippingservice-1.row_count                      30             30  2022-03-21 01:00:00+08:00             3.000000  3.000000  2.000000
1   shippingservice-1       trace.from_frontend-0.row_count                      30             30  2022-03-21 01:00:00+08:00             3.000000  3.000000  2.000000
2       cartservice-2              trace.self.duration_mean                      30             30  2022-03-21 01:00:00+08:00             0.000034  0.000018  1.873278
3         adservice-0      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
4         adservice-0      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
5         adservice-0      trace.from_frontend-2.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
6         adservice-1      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
7         adservice-1      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
8         adservice-1      trace.from_frontend-2.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
9         adservice-2      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
10        adservice-2      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
11        adservice-2      trace.from_frontend-2.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
12       adservice2-0     trace.from_frontend2-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
13      cartservice-0      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
14      cartservice-0      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
15      cartservice-0      trace.from_frontend-2.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
16      cartservice-0               trace.self.duration_p95                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
17      cartservice-0                 trace.self.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
18      cartservice-1      trace.from_frontend-0.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000
19      cartservice-1      trace.from_frontend-1.error_rate                      30             30  2022-03-21 01:00:00+08:00             0.000000  0.000000  0.000000)```
```